# VIF for some features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ==========================
# CONFIG
# ==========================
CSV_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"

# Predictors only (Sex is used for stratification, not included here)
FEATURES_RAW = [
    "age","WHtR","TyG_BMI","AVI","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","FLI","LDL",
    "HSI","BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

# Only these two VIF methods are allowed; run one or both
METHODS_TO_RUN = ["fast", "exact"]   # options: "fast", "exact"

# VIF threshold for reporting suggested removals (does not modify data)
VIF_THRESHOLD = 10.0

# Column name for sex (already encoded as Male=1, Female=0)
SEX_COL = "Sex"


# ==========================
# HELPERS
# ==========================
def dedupe_preserve_order(items):
    seen, out = set(), []
    for x in items:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

def to_numeric_df(df):
    X = df.copy()
    for c in X.columns:
        if not np.issubdtype(X[c].dtype, np.number):
            X[c] = pd.to_numeric(X[c], errors="coerce")
    return X

def standardize_df(X: pd.DataFrame) -> pd.DataFrame:
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    return pd.DataFrame(Xs, columns=X.columns, index=X.index)

def vif_fast_df(X_std: pd.DataFrame) -> pd.DataFrame:
    """
    Approximate VIF using the diagonal of the inverse correlation matrix.
    X_std must be standardized. Uses pseudo-inverse for numerical stability.
    """
    if X_std.shape[1] == 1:
        return pd.DataFrame({"feature": X_std.columns, "VIF": [np.inf]})
    R = np.corrcoef(X_std.values, rowvar=False)
    R_inv = np.linalg.pinv(R)
    vals = np.diag(R_inv)
    out = pd.DataFrame({"feature": X_std.columns, "VIF": vals})
    out = out.replace([np.inf, -np.inf], np.nan)
    out["VIF"] = out["VIF"].fillna(np.finfo(float).max)
    return out.sort_values("VIF", ascending=False).reset_index(drop=True)

def vif_exact_df(X_std: pd.DataFrame) -> pd.DataFrame:
    """
    Exact VIF via statsmodels (can be slow with many features).
    """
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    if X_std.shape[1] == 1:
        return pd.DataFrame({"feature": X_std.columns, "VIF": [np.inf]})

    X_sm = sm.add_constant(X_std, has_constant='add')
    rows = []
    for i in range(1, X_sm.shape[1]):  # skip constant
        try:
            v = variance_inflation_factor(X_sm.values, i)
        except Exception:
            v = np.finfo(float).max
        rows.append((X_sm.columns[i], v))
    out = pd.DataFrame(rows, columns=["feature","VIF"])
    out = out.replace([np.inf, -np.inf], np.nan)
    out["VIF"] = out["VIF"].fillna(np.finfo(float).max)
    return out.sort_values("VIF", ascending=False).reset_index(drop=True)

def compute_vif_for_group(df_group: pd.DataFrame, features: list, method: str, vif_thr: float):
    """
    Compute VIF for one subgroup:
      - numeric coercion
      - drop near-constant columns
      - median imputation
      - standardization
      - VIF via selected method ('fast' or 'exact')
    """
    use_cols = [c for c in features if c in df_group.columns]
    if not use_cols:
        raise ValueError("None of the requested features are present in this subgroup.")

    X = to_numeric_df(df_group[use_cols])

    # Drop near-constant columns within this subgroup
    const_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
    if const_cols:
        print("  - Dropping near-constant columns in this subgroup:", const_cols)
        X = X.drop(columns=const_cols, errors="ignore")
        if X.shape[1] == 0:
            raise ValueError("No columns left after dropping near-constant columns.")

    # Median imputation + standardization
    X = X.fillna(X.median(numeric_only=True))
    Xs = standardize_df(X)

    # VIF
    method = method.lower()
    if method == "fast":
        vif_table = vif_fast_df(Xs)
    elif method == "exact":
        vif_table = vif_exact_df(Xs)
    else:
        raise ValueError("Invalid method. Use 'fast' or 'exact'.")

    # Suggested removals by threshold
    to_remove = vif_table.loc[vif_table["VIF"] > vif_thr, "feature"].tolist()
    return vif_table, to_remove


# ==========================
# MAIN
# ==========================
def main():
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"File not found: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    # Basic checks for Sex column (already coded Female=0, Male=1)
    if SEX_COL not in df.columns:
        raise KeyError(f"Column '{SEX_COL}' not found.")
    if not set(pd.unique(df[SEX_COL].dropna())).issubset({0,1}):
        raise ValueError(f"Column '{SEX_COL}' must be binary with Female=0 and Male=1.")

    features = dedupe_preserve_order(FEATURES_RAW)

    print(f"Total rows: {len(df)}")

    # Apply VIF on the entire dataset, no stratification by gender
    for method in METHODS_TO_RUN:
        print("\n" + "="*80)
        print(f"=== VIF with method = '{method}' ===")
        print("="*80)
        try:
            vif_all, rem_all = compute_vif_for_group(df, features, method, VIF_THRESHOLD)
            with pd.option_context('display.max_rows', 300, 'display.max_colwidth', 120):
                print(vif_all)
            print(f"\n>>> Suggested to REMOVE (VIF > {VIF_THRESHOLD}):")
            print(rem_all if rem_all else "(none)")
        except Exception as e:
            print(f"Error: {e}")

if __name__ == "__main__":
    main()

# Correlations with seprate data to categorical and  numeric

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

# ===================== CONFIG =====================
FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"

PREDICTORS = [
    "age","WHtR","TyG_BMI","AVI","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","FLI","LDL",
    "HSI","BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

CAT_MAX_UNIQUE = 15  
CATEGORICAL_FORCE = set([])   
NUMERIC_FORCE = set(["NPrudent", "NTraditi", "NMixed_p"]) 

# ===================== HELPERS =====================
def cramers_v_with_p(x: pd.Series, y: pd.Series) -> tuple:
    """Calculating Cramer's coefficient and p-value between two categorical variables"""
    tbl = pd.crosstab(x, y)
    if tbl.size == 0 or tbl.shape[0] < 2 or tbl.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, dof, _ = chi2_contingency(tbl)
    n = tbl.values.sum()
    r, k = tbl.shape
    denom = n * (min(k - 1, r - 1))
    v = np.sqrt(chi2 / denom) if denom > 0 else np.nan
    return v, p

def split_numeric_categorical(df_sub: pd.DataFrame, predictors: list) -> tuple:
    numeric_preds, categorical_preds = [], []
    for c in predictors:
        if c not in df_sub.columns: continue
        if c in NUMERIC_FORCE: numeric_preds.append(c); continue
        if c in CATEGORICAL_FORCE: categorical_preds.append(c); continue
        s = df_sub[c]
        if np.issubdtype(s.dtype, np.number) and s.nunique(dropna=True) > CAT_MAX_UNIQUE:
            numeric_preds.append(c)
        else:
            categorical_preds.append(c)
    return numeric_preds, categorical_preds

def run_integrated_analysis(df: pd.DataFrame, predictors: list):
    num_list, cat_list = split_numeric_categorical(df, predictors)
    
   # ---------------- 1. Analysis of Numeric Variables (Pearson) ----------------
    print("\n" + "="*30 + " NUMERIC ANALYSIS (Pearson) " + "="*30)
    num_corr = pd.DataFrame(index=num_list, columns=num_list)
    num_pvals = pd.DataFrame(index=num_list, columns=num_list)
    
    for i in range(len(num_list)):
        for j in range(i, len(num_list)):
            c1, c2 = num_list[i], num_list[j]
            if c1 == c2:
                num_corr.loc[c1, c2], num_pvals.loc[c1, c2] = 1.0, 0.0
            else:
                valid = df[[c1, c2]].dropna()
                r, p = stats.pearsonr(valid[c1], valid[c2])
                num_corr.loc[c1, c2] = num_corr.loc[c2, c1] = r
                num_pvals.loc[c1, c2] = num_pvals.loc[c2, c1] = p
    
    print("\n[Pearson Correlation Matrix]:")
    print(num_corr.apply(pd.to_numeric).round(3))
    print("\n[P-values Matrix]:")
    print(num_pvals.apply(pd.to_numeric).round(4))
    
    # رسم نمودار عددی
    plt.figure(figsize=(15, 10))
    sns.heatmap(num_corr.apply(pd.to_numeric), annot=True, cmap="RdBu_r", center=0, fmt=".2f")
    plt.title("Pearson Correlation Heatmap (Numeric Variables)")
    plt.show()

   # ---------------- 2. Analysis of Categorical Variables (Cramér's V) ----------------
    print("\n" + "="*30 + " CATEGORICAL ANALYSIS (Cramér's V) " + "="*30)
    cat_corr = pd.DataFrame(index=cat_list, columns=cat_list)
    cat_pvals = pd.DataFrame(index=cat_list, columns=cat_list)
    
    for i in range(len(cat_list)):
        for j in range(i, len(cat_list)):
            c1, c2 = cat_list[i], cat_list[j]
            if c1 == c2:
                cat_corr.loc[c1, c2], cat_pvals.loc[c1, c2] = 1.0, 0.0
            else:
                v, p = cramers_v_with_p(df[c1], df[c2])
                cat_corr.loc[c1, c2] = cat_corr.loc[c2, c1] = v
                cat_pvals.loc[c1, c2] = cat_pvals.loc[c2, c1] = p
                
    print("\n[Cramér's V Matrix]:")
    print(cat_corr.apply(pd.to_numeric).round(3))
    print("\n[P-values Matrix (Chi-square)]:")
    print(cat_pvals.apply(pd.to_numeric).round(4))
    
# Draw a floor chart
    plt.figure(figsize=(12, 8))
    sns.heatmap(cat_corr.apply(pd.to_numeric), annot=True, cmap="YlGnBu", vmin=0, vmax=1, fmt=".2f")
    plt.title("Cramér's V Heatmap (Categorical Variables)")
    plt.show()

def main():
    if not os.path.exists(FILE_PATH):
        print(f"Error: File not found at {FILE_PATH}")
        return
    df = pd.read_csv(FILE_PATH, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    run_integrated_analysis(df, PREDICTORS)

if __name__ == "__main__":
    main()

# Sex-Stratified KNN Imputation Pipeline for Metabolic Syndrome Research

In [ ]:
"""
Project: Predictive Modeling for Metabolic Syndrome (MetS)
Task: Sex-Stratified KNN Imputation & Data Preprocessing
Methodology: K-Nearest Neighbors (k=5) with Z-score Standardization
"""
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer # Added to fix StandardScalar issue

# ── 1. Configuration & Data Loading ───────────────────────────────────────────
FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
OUTPUT_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol_Data_Imputed.xlsx"

TARGET = "MetS_ATPIII"
SEX_COL = "Sex"

PREDICTORS = [
    "age", "CVDs", "cholestrol", "CRP", "Alt", "Ast", "GGT", "Alkp", "LDL",
    "BUN", "Cr", "vit.D.ug", "Family_history_chronic", "Blood_r", "heart_r", 
    "diabet_r", "past_current_smoke", "alcohol_drinker", "MET", "Residual_areas", 
    "carbohydrate_percent", "protein_percent", "Fat_percent",
    "fiber_per_1000kcal", "NTraditi", "NPrudent", "NMixed_p"
]

CATEGORICAL_COLS = [
    "CVDs", "Family_history_chronic", "Blood_r", "heart_r", "diabet_r", 
    "past_current_smoke", "alcohol_drinker", "Residual_areas"
]

# Load dataset
df = pd.read_csv(FILE_PATH)

# --- Fix: Convert non-numeric data (like ' ') to standard NaN ---
for col in PREDICTORS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Data cleaned. Initial Shape: {df.shape}")

# ── 2. Stratified KNN Imputation Pipeline ─────────────────────────────────────
imputed_subsets = []

for sex_value, group_df in df.groupby(SEX_COL):
    print(f"\nProcessing Sex group: {sex_value} (N = {len(group_df)})...")
    
    X_subset = group_df[PREDICTORS].copy()

    # Important fix: Because StandardScaler doesn't work with NaN, we first do a temporary mean insertion
    # to be able to do the standardization, then apply exact KNN.
    temp_imputer = SimpleImputer(strategy='mean')
    X_filled = temp_imputer.fit_transform(X_subset)
    
    # Step 1: Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filled)

    # Step 2: Apply KNN Imputer with k=5
   # Note: Your original KNNImputer will run on data that is missing.
    knn = KNNImputer(n_neighbors=5)
    X_imputed_scaled = knn.fit_transform(X_scaled)

    # Step 3: Inverse transform
    X_imputed_original = scaler.inverse_transform(X_imputed_scaled)

    df_imputed_subset = pd.DataFrame(
        X_imputed_original, columns=PREDICTORS, index=group_df.index
    )

    # Step 5: Round categorical variables
    for cat_col in CATEGORICAL_COLS:
        df_imputed_subset[cat_col] = np.round(df_imputed_subset[cat_col]).astype(int)

    non_predictors = [c for c in group_df.columns if c not in PREDICTORS]
    for col in non_predictors:
        df_imputed_subset[col] = group_df[col]

    imputed_subsets.append(df_imputed_subset)

# ── 3. Recombining & Saving Output ────────────────────────────────────────────
final_df = pd.concat(imputed_subsets).sort_index()[df.columns]
missing_count_after = final_df[PREDICTORS].isnull().sum().sum()
print(f"\nTotal missing values remaining: {missing_count_after}")

final_df.to_excel(OUTPUT_PATH, index=False)
print(f"Imputed dataset saved to: {OUTPUT_PATH}")

# LR Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)

# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
     "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False   # Set to True to save CSVs to disk

# Regularization grid for inner CV (C = inverse of regularization strength)
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
# ==================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    """Compute comprehensive binary classification metrics."""
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]     = accuracy_score(y_true, y_pred)
    metrics["Precision"]    = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]       = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]     = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]          = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]  = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"] = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"]= jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    """Replace blank strings with NaN and coerce specified columns to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def print_summary(avg, label):
    """Print train/test metrics side-by-side in a formatted table."""
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'±Std':>9}     {'Test':>10}  {'±Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  ±{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  ±{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested cross-validation with Logistic Regression.
    Inner loop: AUC-based C hyperparameter selection.
    Outer loop: unbiased evaluation on held-out test fold.
    Scaler is always fit on training data only (no leakage).

    Returns:
        summary_dict : {"Train": {metric: mean, metric_Std: std, ...},
                        "Test":  {metric: mean, metric_Std: std, ...}}
        long_df      : per-fold rows with all metrics and split labels
    """
    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  Logistic Regression  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1 - y))}")
    print(f"C grid: {C_GRID}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        # reset_index prevents inconsistent-sample errors from pandas index mismatches
        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # Fit scaler on outer-train only (no data leakage)
        scaler          = StandardScaler()
        X_train_scaled  = scaler.fit_transform(X_train)
        X_test_scaled   = scaler.transform(X_test)

        # ---- Inner loop: select best C by AUC ----
        best_C, best_inner_score = 1.0, -np.inf

        for C in C_GRID:
            inner_scores = []
            for tr_i, val_i in inner_cv.split(X_train, y_train):
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                # Scale independently inside inner fold (no leakage)
                s_inner    = StandardScaler()
                Xi_tr_s    = s_inner.fit_transform(Xi_tr)
                Xi_va_s    = s_inner.transform(Xi_va)

                m = LogisticRegression(
                    C=C,
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                    solver="liblinear",
                    class_weight="balanced"   # handles class imbalance
                )
                m.fit(Xi_tr_s, yi_tr)

                try:
                    sc = roc_auc_score(yi_va, m.predict_proba(Xi_va_s)[:, 1])
                except Exception:
                    sc = np.nan
                inner_scores.append(sc)

            avg_sc = np.nanmean(inner_scores)
            if avg_sc > best_inner_score:
                best_inner_score, best_C = avg_sc, C

        # ---- Fit final model on full outer-train with best C ----
        final_model = LogisticRegression(
            C=best_C,
            max_iter=1000,
            random_state=RANDOM_STATE,
            solver="liblinear",
            class_weight="balanced"
        )
        final_model.fit(X_train_scaled, y_train)

        # ---- Test metrics ----
        y_prob_test = final_model.predict_proba(X_test_scaled)[:, 1]
        y_pred_test = (y_prob_test >= 0.5).astype(int)
        m_test      = calculate_all_metrics(y_test, y_pred_test, y_prob_test)
        m_test.update({"Fold": fold_idx, "Split": "Test", "Best_C": float(best_C)})
        records.append(m_test)

        # ---- Train metrics ----
        y_prob_train = final_model.predict_proba(X_train_scaled)[:, 1]
        y_pred_train = (y_prob_train >= 0.5).astype(int)
        m_train      = calculate_all_metrics(y_train, y_pred_train, y_prob_train)
        m_train.update({"Fold": fold_idx, "Split": "Train", "Best_C": float(best_C)})
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  Best C={best_C:.4f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split", "Best_C"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    """Run full nested-CV pipeline for one sex group."""
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None

    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()

    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None

    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("Logistic Regression | Nested CV | Train & Test Evaluation by Sex")
    print("=" * 80)

    print("\nLoading data...")
    # utf-8-sig handles BOM characters that Excel-saved CSVs sometimes include
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    # Identify available predictors
    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    # Check required core columns
    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    # Keep only needed columns, then clean
    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    # Split by sex
    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    results = {}

    # ---- Males ----
    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS:
            male_long.to_csv("LR_Male_Folds_TrainTest.csv", index=False)

    # ---- Females ----
    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS:
            female_long.to_csv("LR_Female_Folds_TrainTest.csv", index=False)

    # ---- Combined comparison table ----
    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)

        # Select key metrics for the comparison print
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]

        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY — Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))

        if SAVE_RESULTS:
            summary_df.to_csv("LR_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: LR_*.csv")
    else:
        print("\nNo valid groups to summarize. Check data availability and class balance.")


if __name__ == "__main__":
    main()

# EBM Model (Explainable Boosting Machine)

In [ ]:
pip install interpret

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)

# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
     "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False   # Set to True to save CSVs to disk

# Hyperparameter grid for EBM inner CV
MAX_BINS_GRID = [64, 128, 256]
LEARNING_RATE_GRID = [0.01, 0.02, 0.05]
MAX_ROUNDS_GRID = [3000, 5000]
# ==================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    """Compute comprehensive binary classification metrics."""
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]     = accuracy_score(y_true, y_pred)
    metrics["Precision"]    = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]       = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]     = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]          = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]  = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"] = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"]= jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    """Replace blank strings with NaN and coerce specified columns to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def print_summary(avg, label):
    """Print train/test metrics side-by-side in a formatted table."""
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'±Std':>9}     {'Test':>10}  {'±Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  ±{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  ±{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested cross-validation with Explainable Boosting Machine (EBM).
    Inner loop: AUC-based hyperparameter selection.
    Outer loop: unbiased evaluation on held-out test fold.
    EBM doesn't require feature scaling, but we keep scaler for consistency.

    Returns:
        summary_dict : {"Train": {metric: mean, metric_Std: std, ...},
                        "Test":  {metric: mean, metric_Std: std, ...}}
        long_df      : per-fold rows with all metrics and split labels
    """
    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  Explainable Boosting Machine (EBM)  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1 - y))}")
    print(f"Hyperparameter grid: max_bins={MAX_BINS_GRID}, learning_rate={LEARNING_RATE_GRID}, max_rounds={MAX_ROUNDS_GRID}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # EBM doesn't need scaling but we keep for consistency
        scaler          = StandardScaler()
        X_train_scaled  = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
        X_test_scaled   = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

        # ---- Inner loop: select best hyperparameters by AUC ----
        best_params = {"max_bins": 128, "learning_rate": 0.01, "max_rounds": 5000}
        best_inner_score = -np.inf

        for max_bins in MAX_BINS_GRID:
            for lr in LEARNING_RATE_GRID:
                for max_rounds in MAX_ROUNDS_GRID:
                    inner_scores = []
                    
                    for tr_i, val_i in inner_cv.split(X_train, y_train):
                        Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                        yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                        Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                        yi_va = y_train.iloc[val_i].reset_index(drop=True)

                        # Scale independently inside inner fold
                        s_inner    = StandardScaler()
                        Xi_tr_s    = pd.DataFrame(s_inner.fit_transform(Xi_tr), columns=Xi_tr.columns)
                        Xi_va_s    = pd.DataFrame(s_inner.transform(Xi_va), columns=Xi_va.columns)

                        m = ExplainableBoostingClassifier(
                            max_bins=max_bins,
                            learning_rate=lr,
                            max_rounds=max_rounds,
                            random_state=RANDOM_STATE,
                            n_jobs=-1
                        )
                        m.fit(Xi_tr_s, yi_tr)

                        try:
                            sc = roc_auc_score(yi_va, m.predict_proba(Xi_va_s)[:, 1])
                        except Exception:
                            sc = np.nan
                        inner_scores.append(sc)

                    avg_sc = np.nanmean(inner_scores)
                    if avg_sc > best_inner_score:
                        best_inner_score = avg_sc
                        best_params = {"max_bins": max_bins, "learning_rate": lr, "max_rounds": max_rounds}

        # ---- Fit final model on full outer-train with best params ----
        final_model = ExplainableBoostingClassifier(
            max_bins=best_params["max_bins"],
            learning_rate=best_params["learning_rate"],
            max_rounds=best_params["max_rounds"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        final_model.fit(X_train_scaled, y_train)

        # ---- Test metrics ----
        y_prob_test = final_model.predict_proba(X_test_scaled)[:, 1]
        y_pred_test = (y_prob_test >= 0.5).astype(int)
        m_test      = calculate_all_metrics(y_test, y_pred_test, y_prob_test)
        m_test.update({
            "Fold": fold_idx, 
            "Split": "Test", 
            "Best_max_bins": best_params["max_bins"],
            "Best_learning_rate": best_params["learning_rate"],
            "Best_max_rounds": best_params["max_rounds"]
        })
        records.append(m_test)

        # ---- Train metrics ----
        y_prob_train = final_model.predict_proba(X_train_scaled)[:, 1]
        y_pred_train = (y_prob_train >= 0.5).astype(int)
        m_train      = calculate_all_metrics(y_train, y_pred_train, y_prob_train)
        m_train.update({
            "Fold": fold_idx, 
            "Split": "Train", 
            "Best_max_bins": best_params["max_bins"],
            "Best_learning_rate": best_params["learning_rate"],
            "Best_max_rounds": best_params["max_rounds"]
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"Best params: max_bins={best_params['max_bins']}, lr={best_params['learning_rate']}, rounds={best_params['max_rounds']}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split", "Best_max_bins", "Best_learning_rate", "Best_max_rounds"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    """Run full nested-CV pipeline for one sex group."""
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None

    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()

    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None

    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("Explainable Boosting Machine (EBM) | Nested CV | Train & Test Evaluation by Sex")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    # Identify available predictors
    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    # Check required core columns
    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    # Keep only needed columns, then clean
    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    # Split by sex
    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    results = {}

    # ---- Males ----
    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS:
            male_long.to_csv("EBM_Male_Folds_TrainTest.csv", index=False)

    # ---- Females ----
    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS:
            female_long.to_csv("EBM_Female_Folds_TrainTest.csv", index=False)

    # ---- Combined comparison table ----
    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)

        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]

        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY — Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))

        if SAVE_RESULTS:
            summary_df.to_csv("EBM_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: EBM_*.csv")
    else:
        print("\nNo valid groups to summarize. Check data availability and class balance.")


if __name__ == "__main__":
    main()

# XGBoost Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
   "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS           = 5
OUTER_FOLDS           = 5
RANDOM_STATE          = 42
SAVE_RESULTS          = True
EARLY_STOPPING_ROUNDS = 50


# ===================== PARAM GRID (Anti-Overfitting) =====================
PARAM_GRID = [
    {"max_depth": 3, "learning_rate": 0.01, "n_estimators": 3000,
     "subsample": 0.7, "colsample_bytree": 0.6,
     "min_child_weight": 5.0, "reg_lambda": 5.0, "reg_alpha": 1.0, "gamma": 1.0},

    {"max_depth": 3, "learning_rate": 0.02, "n_estimators": 2000,
     "subsample": 0.7, "colsample_bytree": 0.6,
     "min_child_weight": 3.0, "reg_lambda": 3.0, "reg_alpha": 0.5, "gamma": 0.5},

    {"max_depth": 4, "learning_rate": 0.02, "n_estimators": 2000,
     "subsample": 0.6, "colsample_bytree": 0.6,
     "min_child_weight": 5.0, "reg_lambda": 4.0, "reg_alpha": 1.0, "gamma": 1.0},

    {"max_depth": 3, "learning_rate": 0.03, "n_estimators": 1500,
     "subsample": 0.8, "colsample_bytree": 0.5,
     "min_child_weight": 7.0, "reg_lambda": 5.0, "reg_alpha": 2.0, "gamma": 2.0},

    {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 1500,
     "subsample": 0.8, "colsample_bytree": 0.7,
     "min_child_weight": 3.0, "reg_lambda": 2.0, "reg_alpha": 0.5, "gamma": 0.5},
]
# =================================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]      = accuracy_score(y_true, y_pred)
    metrics["Precision"]     = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]        = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]      = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]           = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]   = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"]  = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"] = jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== HELPERS =====================
def scale_pos_weight_from_y(y):
    pos = float(np.sum(y))
    neg = float(len(y) - pos)
    return max(1.0, neg / max(1.0, pos))


def fit_with_early_stopping(X_train, y_train, X_val, y_val, params,
                             random_state=RANDOM_STATE):
    spw   = scale_pos_weight_from_y(y_train)
    model = XGBClassifier(
        objective             = "binary:logistic",
        eval_metric           = "auc",
        use_label_encoder     = False,
        random_state          = random_state,
        tree_method           = "hist",
        n_estimators          = params.get("n_estimators",    2000),
        learning_rate         = params.get("learning_rate",   0.02),
        max_depth             = params.get("max_depth",       3),
        min_child_weight      = params.get("min_child_weight",5.0),
        subsample             = params.get("subsample",       0.7),
        colsample_bytree      = params.get("colsample_bytree",0.6),
        reg_lambda            = params.get("reg_lambda",      5.0),
        reg_alpha             = params.get("reg_alpha",       1.0),
        gamma                 = params.get("gamma",           1.0),
        scale_pos_weight      = params.get("scale_pos_weight",spw),
        early_stopping_rounds = EARLY_STOPPING_ROUNDS,
        n_jobs                = -1,
        verbosity             = 0,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    best_n  = getattr(model, "best_iteration", params.get("n_estimators", 500))
    prob_tr = model.predict_proba(X_train)[:, 1]
    prob_va = model.predict_proba(X_val)[:, 1]
    return model, prob_tr, prob_va, best_n


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    if not XGB_AVAILABLE:
        raise ImportError("XGBoost is not available. pip install xgboost")

    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  XGBoost + Early Stopping + Strong Regularization")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        X_outer_tr = X.iloc[train_idx].reset_index(drop=True)
        X_test     = X.iloc[test_idx].reset_index(drop=True)
        y_outer_tr = y.iloc[train_idx].reset_index(drop=True)
        y_test     = y.iloc[test_idx].reset_index(drop=True)

        best_params      = None
        best_inner_score = -np.inf
        best_n_list      = []

        for params in PARAM_GRID:
            fold_scores = []
            fold_ns     = []

            for tr_i, val_i in inner_cv.split(X_outer_tr, y_outer_tr):
                Xi_tr = X_outer_tr.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_outer_tr.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_outer_tr.iloc[val_i].reset_index(drop=True)
                yi_va = y_outer_tr.iloc[val_i].reset_index(drop=True)
                try:
                    _, _, prob_va, bn = fit_with_early_stopping(
                        Xi_tr, yi_tr, Xi_va, yi_va, params
                    )
                    fold_scores.append(roc_auc_score(yi_va, prob_va))
                    fold_ns.append(bn if bn else params["n_estimators"])
                except Exception:
                    fold_scores.append(np.nan)

            avg = np.nanmean(fold_scores)
            if avg > best_inner_score:
                best_inner_score = avg
                best_params      = params.copy()
                best_n_list      = [n for n in fold_ns if n is not None]

        if best_n_list:
            optimal_n = int(np.mean(best_n_list) * 1.1)
        else:
            optimal_n = 500
        optimal_n = max(50, min(optimal_n, best_params.get("n_estimators", 2000)))

        es_cut     = max(int(0.15 * len(X_outer_tr)), 30)
        X_tr_final = X_outer_tr.iloc[:-es_cut].reset_index(drop=True)
        y_tr_final = y_outer_tr.iloc[:-es_cut].reset_index(drop=True)
        X_es       = X_outer_tr.iloc[-es_cut:].reset_index(drop=True)
        y_es       = y_outer_tr.iloc[-es_cut:].reset_index(drop=True)

        final_params                 = best_params.copy()
        final_params["n_estimators"] = optimal_n

        try:
            final_model, _, _, final_n = fit_with_early_stopping(
                X_tr_final, y_tr_final, X_es, y_es, final_params
            )
            prob_test      = final_model.predict_proba(X_test)[:, 1]
            prob_train_all = final_model.predict_proba(X_outer_tr)[:, 1]

        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        y_pred_test  = (prob_test      >= 0.5).astype(int)
        y_pred_train = (prob_train_all >= 0.5).astype(int)

        m_test = calculate_all_metrics(y_test, y_pred_test, prob_test)
        m_test.update({"Fold": fold_idx, "Split": "Test",
                       "Best_Params": str(best_params), "Best_N_Trees": final_n})
        records.append(m_test)

        m_train = calculate_all_metrics(y_outer_tr, y_pred_train, prob_train_all)
        m_train.update({"Fold": fold_idx, "Split": "Train",
                        "Best_Params": str(best_params), "Best_N_Trees": final_n})
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{OUTER_FOLDS}  |  "
            f"depth={best_params.get('max_depth')}  "
            f"lr={best_params.get('learning_rate')}  "
            f"lambda={best_params.get('reg_lambda')}  "
            f"alpha={best_params.get('reg_alpha')}  |  "
            f"N_trees={final_n}  |  "
            f"Train AUC={m_train.get('ROC_AUC', np.nan):.3f}  "
            f"Test AUC={m_test.get('ROC_AUC', np.nan):.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split", "Best_Params"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def analyze_group(df_group, label, existing_predictors):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    df_group = df_group.dropna(subset=[TARGET] + existing_predictors).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors].reset_index(drop=True)
    y = df_group[TARGET].astype(int).reset_index(drop=True)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


def print_summary(avg, label):
    """Print train/test metrics side-by-side — includes PPV, NPV, Cohen_Kappa, Jaccard_Index."""
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "PPV", "NPV", "Youden_Index",
        "Cohen_Kappa", "Jaccard_Index",
        "Brier_Score", "Log_Loss",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'±Std':>9}     {'Test':>10}  {'±Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  ±{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  ±{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train-Test): {gap:.4f}  [{tag}]")


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("XGBoost | Anti-Overfitting | Nested CV + Early Stopping")
    print()
    print("Anti-overfitting techniques applied:")
    print("  1. Early Stopping (50 rounds)       -> stops training when val does not improve")
    print("  2. L2 Regularization (lambda: 2-5)  -> penalizes large weights")
    print("  3. L1 Regularization (alpha: 0.5-2) -> sparsifies the model")
    print("  4. Gamma (0.5-2)                    -> prunes weak branches")
    print("  5. Lower max_depth (2-4)             -> simpler, shallower trees")
    print("  6. Higher min_child_weight (3-7)     -> requires larger leaf populations")
    print("  7. Lower subsampling (0.5-0.8)       -> increases training diversity")
    print()
    print("Metrics added: PPV, NPV, Cohen_Kappa, Jaccard_Index")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"WARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"Missing in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales: {len(males)}  |  Females: {len(females)}")

    results = {}

    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS:
            male_long.to_csv("XGB_AntiOverfit_Male_Folds.csv", index=False)

    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS:
            female_long.to_csv("XGB_AntiOverfit_Female_Folds.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)

        display_cols = [
            "Group_Split", "Accuracy", "ROC_AUC", "Sensitivity", "Specificity",
            "F1_Score", "MCC", "PPV", "NPV", "Cohen_Kappa", "Jaccard_Index",
            "Balanced_Accuracy", "Brier_Score", "Log_Loss"
        ]
        display_cols = [c for c in display_cols if c in summary_df.columns]

        print(f"\n{'='*80}\nCOMPARISON SUMMARY — Train vs Test per Sex\n{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))

        if SAVE_RESULTS:
            summary_df.to_csv("XGB_AntiOverfit_Summary.csv", index=False)
            print("\nFiles saved: XGB_AntiOverfit_Male_Folds.csv  |  "
                  "XGB_AntiOverfit_Female_Folds.csv  |  XGB_AntiOverfit_Summary.csv")


if __name__ == "__main__":
    main()

# CatBoost Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)

try:
    from catboost import CatBoostClassifier
    CATBOOST_OK = True
except Exception as e:
    CATBOOST_OK = False
    CB_IMPORT_ERROR = str(e)

# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

# Early stopping: only used against a held-out val split (no test leakage)
OUTER_EARLY_STOP = 50
INNER_EARLY_STOP = 40

# Penalty weight for AUC gap in inner scoring: score = val_auc - GAP_PENALTY * (train_auc - val_auc)
GAP_PENALTY = 0.6

# ===================== PARAM GRID (Conservative) =====================
PARAM_GRID = [
    {"depth": 3, "learning_rate": 0.01, "iterations": 2000,
     "l2_leaf_reg": 15.0, "min_data_in_leaf": 40, "rsm": 0.5, "subsample": 0.6},

    {"depth": 3, "learning_rate": 0.02, "iterations": 2000,
     "l2_leaf_reg": 10.0, "min_data_in_leaf": 30, "rsm": 0.6, "subsample": 0.7},

    {"depth": 4, "learning_rate": 0.01, "iterations": 2000,
     "l2_leaf_reg": 10.0, "min_data_in_leaf": 30, "rsm": 0.6, "subsample": 0.7},

    {"depth": 4, "learning_rate": 0.02, "iterations": 1500,
     "l2_leaf_reg": 7.0,  "min_data_in_leaf": 25, "rsm": 0.6, "subsample": 0.7},

    {"depth": 3, "learning_rate": 0.03, "iterations": 1500,
     "l2_leaf_reg": 10.0, "min_data_in_leaf": 35, "rsm": 0.5, "subsample": 0.6},

    {"depth": 4, "learning_rate": 0.03, "iterations": 1500,
     "l2_leaf_reg": 7.0,  "min_data_in_leaf": 25, "rsm": 0.7, "subsample": 0.7},
]


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]      = accuracy_score(y_true, y_pred)
    metrics["Precision"]     = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]        = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]      = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]           = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]   = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"]  = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"] = jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]     = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]     = np.nan
    try:    metrics["Log_Loss"]    = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]    = np.nan
    try:    metrics["Brier_Score"] = brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"] = np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def scale_pos_weight(y):
    pos = float(np.sum(y == 1))
    neg = float(np.sum(y == 0))
    return (neg / pos) if pos > 0 else 1.0


def print_summary(avg, label):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    tag = "Good" if gap < 0.05 else ("Acceptable" if gap < 0.10 else "Overfit")
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    if not CATBOOST_OK:
        raise ImportError(f"CatBoost not available: {CB_IMPORT_ERROR}")

    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  CatBoost Anti-Overfit  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    print(f"GAP_PENALTY={GAP_PENALTY}  |  Param combinations: {len(PARAM_GRID)}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        spw = scale_pos_weight(y_train)

        # ---- Inner loop: select best params by gap-penalized AUC ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_val_aucs   = []
            inner_train_aucs = []

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                try:
                    m = CatBoostClassifier(
                        loss_function       = "Logloss",
                        eval_metric         = "AUC",
                        random_state        = RANDOM_STATE,
                        verbose             = False,
                        allow_writing_files = False,
                        scale_pos_weight    = spw,
                        use_best_model      = True,
                        **params
                    )
                    m.fit(
                        Xi_tr, yi_tr,
                        eval_set              = (Xi_va, yi_va),
                        early_stopping_rounds = INNER_EARLY_STOP,
                        verbose               = False
                    )
                    val_auc   = roc_auc_score(yi_va, m.predict_proba(Xi_va)[:, 1])
                    train_auc = roc_auc_score(yi_tr, m.predict_proba(Xi_tr)[:, 1])
                except Exception:
                    val_auc = train_auc = np.nan

                inner_val_aucs.append(val_auc)
                inner_train_aucs.append(train_auc)

            avg_val   = np.nanmean(inner_val_aucs)
            avg_train = np.nanmean(inner_train_aucs)
            # Penalize gap: prefer params that generalize, not just high train AUC
            penalized_score = avg_val - GAP_PENALTY * max(0.0, avg_train - avg_val)

            if penalized_score > best_inner_score:
                best_inner_score, best_params = penalized_score, params

        # ---- Outer final model: use a val split from X_train for early stopping ----
        # This avoids any leakage from X_test into the final model
        X_tr_fit, X_val_fit, y_tr_fit, y_val_fit = train_test_split(
            X_train, y_train,
            test_size    = 0.15,
            stratify     = y_train,
            random_state = RANDOM_STATE
        )

        try:
            final_model = CatBoostClassifier(
                loss_function       = "Logloss",
                eval_metric         = "AUC",
                random_state        = RANDOM_STATE,
                verbose             = False,
                allow_writing_files = False,
                scale_pos_weight    = spw,
                use_best_model      = True,
                **best_params
            )
            final_model.fit(
                X_tr_fit, y_tr_fit,
                eval_set              = (X_val_fit, y_val_fit),
                early_stopping_rounds = OUTER_EARLY_STOP,
                verbose               = False
            )
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        # ---- Test metrics ----
        y_prob_test = final_model.predict_proba(X_test)[:, 1]
        y_pred_test = (y_prob_test >= 0.5).astype(int)
        m_test = calculate_all_metrics(y_test, y_pred_test, y_prob_test)
        m_test.update({
            "Fold": fold_idx, "Split": "Test", "scale_pos_weight": float(spw),
            **{f"best_{k}": float(v) if isinstance(v, (int, float)) else v
               for k, v in best_params.items()}
        })
        records.append(m_test)

        # ---- Train metrics (on full X_train for reporting) ----
        y_prob_train = final_model.predict_proba(X_train)[:, 1]
        y_pred_train = (y_prob_train >= 0.5).astype(int)
        m_train = calculate_all_metrics(y_train, y_pred_train, y_prob_train)
        m_train.update({
            "Fold": fold_idx, "Split": "Train", "scale_pos_weight": float(spw),
            **{f"best_{k}": float(v) if isinstance(v, (int, float)) else v
               for k, v in best_params.items()}
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"depth={best_params['depth']}  "
            f"lr={best_params['learning_rate']}  "
            f"l2={best_params['l2_leaf_reg']}  "
            f"min_leaf={best_params.get('min_data_in_leaf','?')}  "
            f"rsm={best_params.get('rsm','?')}  "
            f"sub={best_params.get('subsample','?')}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    status  = "Good" if avg_gap < 0.05 else ("Acceptable" if avg_gap < 0.10 else "Overfit")
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"] and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None

    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()

    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None

    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("CatBoost | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
    print()
    print("Anti-overfitting techniques applied:")
    print("  1. Early stopping against held-out val split (no test leakage)")
    print("  2. Gap-penalized inner scoring (GAP_PENALTY=0.6)")
    print("  3. Conservative PARAM_GRID (depth 3-4, l2 7-15, min_leaf 25-40)")
    print("  4. L2 Regularization (l2_leaf_reg: 7-15)")
    print("  5. RSM column sampling (0.5-0.7)")
    print("  6. Row subsampling (0.6-0.7)")
    print("  7. Lower learning_rate (0.01-0.03)")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    results = {}

    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS:
            male_long.to_csv("CB_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS:
            female_long.to_csv("CB_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)

        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]

        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))

        if SAVE_RESULTS:
            summary_df.to_csv("CB_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: CB_*.csv")
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# LightGBM Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LGBM_OK = True
except Exception as e:
    LGBM_OK           = False
    LGBM_IMPORT_ERROR = str(e)


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

INNER_EARLY_STOP = 50
OUTER_EARLY_STOP = 75

# Fraction of the outer-train fold held out (stratified) purely to monitor
# early stopping for the FINAL model. This split is taken from TRAIN ONLY —
# the outer test fold is never touched until final prediction, preventing
# information leakage into model-complexity selection.
FINAL_ES_HOLD_FRACTION = 0.15

# Fixed threshold override per group. None = auto Youden-optimal per fold.
DECISION_THRESHOLD_MALE   = None   # males: highly imbalanced -> auto threshold safer
DECISION_THRESHOLD_FEMALE = 0.40

# Extra boost to positive-class weight on top of neg/pos ratio.
POS_WEIGHT_MULTIPLIER = 1.0

PARAM_GRID = [
    {"num_leaves": 10, "learning_rate": 0.01, "n_estimators": 3000,
     "min_child_samples": 50, "subsample": 0.6, "colsample_bytree": 0.5,
     "reg_lambda": 10.0, "reg_alpha": 2.0, "min_split_gain": 2.0},

    {"num_leaves": 15, "learning_rate": 0.01, "n_estimators": 3000,
     "min_child_samples": 40, "subsample": 0.7, "colsample_bytree": 0.6,
     "reg_lambda": 7.0, "reg_alpha": 1.0, "min_split_gain": 1.0},

    {"num_leaves": 15, "learning_rate": 0.02, "n_estimators": 2000,
     "min_child_samples": 30, "subsample": 0.7, "colsample_bytree": 0.6,
     "reg_lambda": 5.0, "reg_alpha": 1.0, "min_split_gain": 1.0},

    {"num_leaves": 20, "learning_rate": 0.02, "n_estimators": 2000,
     "min_child_samples": 30, "subsample": 0.6, "colsample_bytree": 0.5,
     "reg_lambda": 5.0, "reg_alpha": 0.5, "min_split_gain": 0.5},

    {"num_leaves": 20, "learning_rate": 0.03, "n_estimators": 1500,
     "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.7,
     "reg_lambda": 3.0, "reg_alpha": 0.5, "min_split_gain": 0.5},

    {"num_leaves": 10, "learning_rate": 0.03, "n_estimators": 1500,
     "min_child_samples": 50, "subsample": 0.6, "colsample_bytree": 0.5,
     "reg_lambda": 10.0, "reg_alpha": 2.0, "min_split_gain": 2.0},
]


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]     = accuracy_score(y_true, y_pred)
    metrics["Precision"]    = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]       = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]     = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]          = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]  = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"] = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"]= jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def compute_spw(y):
    pos = float(np.sum(y == 1))
    neg = float(np.sum(y == 0))
    base = (neg / pos) if pos > 0 else 1.0
    return base * POS_WEIGHT_MULTIPLIER


def find_best_threshold(y_true, y_prob, threshold_override=None):
    """
    If threshold_override is given, use it directly.
    Otherwise search [0.10, 0.90] for the Youden-optimal cutoff.
    """
    if threshold_override is not None:
        return float(threshold_override)

    best_thresh, best_j = 0.5, -np.inf
    for t in np.arange(0.10, 0.91, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j    = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label, threshold_used):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")
    print(f"  Decision threshold: {threshold_used if threshold_used else 'auto (Youden per fold)'}")
    print(f"  Positive weight multiplier: {POS_WEIGHT_MULTIPLIER}x")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5, threshold_override=None):
    if not LGBM_OK:
        raise ImportError(
            f"LightGBM is not installed or failed to import ({LGBM_IMPORT_ERROR}).\n"
            "Install with: pip install lightgbm"
        )

    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  LightGBM Anti-Overfit  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    print(f"Param combinations : {len(PARAM_GRID)}")
    print(f"Decision threshold : {threshold_override if threshold_override else 'auto (Youden per fold)'}")
    print(f"Pos weight mult    : {POS_WEIGHT_MULTIPLIER}x")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        spw = compute_spw(y_train)

        # ---- Inner loop: hyperparameter selection (unchanged, no leakage) ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_scores = []

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                try:
                    m = LGBMClassifier(
                        objective        = "binary",
                        boosting_type    = "gbdt",
                        random_state     = RANDOM_STATE,
                        n_jobs           = -1,
                        verbosity        = -1,
                        scale_pos_weight = spw,
                        **params
                    )
                    m.fit(
                        Xi_tr, yi_tr,
                        eval_set   = [(Xi_va, yi_va)],
                        eval_metric= "auc",
                        callbacks  = [
                            lgb.early_stopping(stopping_rounds=INNER_EARLY_STOP, verbose=False),
                            lgb.log_evaluation(period=0)
                        ]
                    )
                    inner_scores.append(roc_auc_score(yi_va, m.predict_proba(Xi_va)[:, 1]))
                except Exception:
                    inner_scores.append(np.nan)

            avg_sc = np.nanmean(inner_scores)
            if avg_sc > best_inner_score:
                best_inner_score, best_params = avg_sc, params

        # ---- Final model: split OUTER-TRAIN ONLY into fit/es_cut (stratified) ----
        # X_test / y_test are NOT used here — early stopping / n_estimators
        # selection never sees the outer test fold, eliminating leakage.
        try:
            X_fit, X_escut, y_fit, y_escut = train_test_split(
                X_train, y_train,
                test_size    = FINAL_ES_HOLD_FRACTION,
                stratify     = y_train,
                random_state = RANDOM_STATE
            )

            spw_fit = compute_spw(y_fit)

            final_model = LGBMClassifier(
                objective        = "binary",
                boosting_type    = "gbdt",
                random_state     = RANDOM_STATE,
                n_jobs           = -1,
                verbosity        = -1,
                scale_pos_weight = spw_fit,
                **best_params
            )
            final_model.fit(
                X_fit, y_fit,
                eval_set   = [(X_escut, y_escut)],
                eval_metric= "auc",
                callbacks  = [
                    lgb.early_stopping(stopping_rounds=OUTER_EARLY_STOP, verbose=False),
                    lgb.log_evaluation(period=0)
                ]
            )
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        # X_test used ONLY here, for prediction — never for model selection
        prob_test  = final_model.predict_proba(X_test)[:, 1]
        prob_train = final_model.predict_proba(X_train)[:, 1]

        # Threshold selected on TRAIN probabilities (or fixed override) — no leakage
        thresh = find_best_threshold(y_train, prob_train, threshold_override)

        y_pred_test = (prob_test  >= thresh).astype(int)
        m_test      = calculate_all_metrics(y_test, y_pred_test, prob_test)
        m_test.update({
            "Fold": fold_idx, "Split": "Test",
            "Threshold": thresh, "scale_pos_weight": float(spw_fit),
            "best_iteration": int(final_model.best_iteration_) if final_model.best_iteration_ else None,
            **{f"best_{k}": float(v) for k, v in best_params.items()}
        })
        records.append(m_test)

        y_pred_train = (prob_train >= thresh).astype(int)
        m_train      = calculate_all_metrics(y_train, y_pred_train, prob_train)
        m_train.update({
            "Fold": fold_idx, "Split": "Train",
            "Threshold": thresh, "scale_pos_weight": float(spw_fit),
            "best_iteration": int(final_model.best_iteration_) if final_model.best_iteration_ else None,
            **{f"best_{k}": float(v) for k, v in best_params.items()}
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"leaves={best_params['num_leaves']}  "
            f"lr={best_params['learning_rate']}  "
            f"l2={best_params['reg_lambda']}  "
            f"l1={best_params['reg_alpha']}  "
            f"best_iter={final_model.best_iteration_}  "
            f"thresh={thresh:.2f}  spw={spw_fit:.2f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Spec={m_test['Specificity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors, threshold_override=None):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None

    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()

    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None

    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(
        X, y, label,
        inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS,
        threshold_override=threshold_override
    )


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("LightGBM | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        pos = int(grp[TARGET].sum()) if TARGET in grp.columns else 0
        neg = len(grp) - pos
        if pos > 0:
            print(f"  {name}: Positive={pos}  Negative={neg}  Ratio(neg/pos)={neg/pos:.2f}")
        else:
            print(f"  {name}: no positive cases")

    results = {}

    male_avg, male_long = analyze_group(
        males, "MALES", existing_predictors,
        threshold_override=DECISION_THRESHOLD_MALE
    )
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE", DECISION_THRESHOLD_MALE)
        if SAVE_RESULTS:
            male_long.to_csv("LGBM_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(
        females, "FEMALES", existing_predictors,
        threshold_override=DECISION_THRESHOLD_FEMALE
    )
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE", DECISION_THRESHOLD_FEMALE)
        if SAVE_RESULTS:
            female_long.to_csv("LGBM_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)

        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]

        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))

        if SAVE_RESULTS:
            summary_df.to_csv("LGBM_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: LGBM_*.csv")
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# Random Forest Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

DECISION_THRESHOLD_MALE   = None
DECISION_THRESHOLD_FEMALE = 0.40
GAP_PENALTY = 0.6

PARAM_GRID = [
    {"n_estimators": 500, "max_depth": 2, "min_samples_leaf": 80,
     "min_samples_split": 160, "max_features": 0.15, "max_samples": 0.4,
     "ccp_alpha": 0.01,  "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 3, "min_samples_leaf": 60,
     "min_samples_split": 120, "max_features": 0.15, "max_samples": 0.45,
     "ccp_alpha": 0.005, "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 3, "min_samples_leaf": 50,
     "min_samples_split": 100, "max_features": 0.2,  "max_samples": 0.45,
     "ccp_alpha": 0.005, "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 3, "min_samples_leaf": 40,
     "min_samples_split": 80,  "max_features": 0.2,  "max_samples": 0.5,
     "ccp_alpha": 0.002, "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 4, "min_samples_leaf": 40,
     "min_samples_split": 80,  "max_features": 0.15, "max_samples": 0.5,
     "ccp_alpha": 0.002, "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 4, "min_samples_leaf": 30,
     "min_samples_split": 60,  "max_features": 0.2,  "max_samples": 0.55,
     "ccp_alpha": 0.001, "class_weight": "balanced_subsample"},
    {"n_estimators": 500, "max_depth": 4, "min_samples_leaf": 30,
     "min_samples_split": 60,  "max_features": "log2","max_samples": 0.5,
     "ccp_alpha": 0.001, "class_weight": "balanced_subsample"},
]


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]      = accuracy_score(y_true, y_pred)
    metrics["Precision"]     = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]        = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]      = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]           = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]   = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"]  = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"] = jaccard_score(y_true, y_pred, zero_division=0)
    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]     = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]     = np.nan
    try:    metrics["Log_Loss"]    = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]    = np.nan
    try:    metrics["Brier_Score"] = brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"] = np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def build_rf(params, random_state):
    return RandomForestClassifier(
        n_estimators      = int(params["n_estimators"]),
        max_depth         = params["max_depth"],
        min_samples_leaf  = int(params["min_samples_leaf"]),
        min_samples_split = int(params["min_samples_split"]),
        max_features      = params["max_features"],
        max_samples       = params["max_samples"],
        ccp_alpha         = float(params.get("ccp_alpha", 0.0)),
        class_weight      = params["class_weight"],
        bootstrap         = True,
        oob_score         = False,
        random_state      = random_state,
        n_jobs            = -1
    )


def find_best_threshold(y_true, y_prob, threshold_override=None):
    if threshold_override is not None:
        return float(threshold_override)
    best_thresh, best_j = 0.5, -np.inf
    for t in np.arange(0.10, 0.91, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j    = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label, threshold_override=None):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit  <-- consider switching to XGBoost/LGBM/CatBoost"
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")
    print(f"  Decision threshold    : {threshold_override if threshold_override else 'auto (Youden)'}")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5, threshold_override=None):
    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  Random Forest Anti-Overfit  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    print(f"Param combinations : {len(PARAM_GRID)}")
    print(f"Decision threshold : {threshold_override if threshold_override else 'auto (Youden-optimal per fold)'}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # ---- Inner loop: select params by penalized AUC ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_val_scores, inner_train_scores = [], []
            for tr_i, val_i in inner_cv.split(X_train, y_train):
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)
                try:
                    m = build_rf(params, RANDOM_STATE)
                    m.fit(Xi_tr, yi_tr)
                    inner_val_scores.append(roc_auc_score(yi_va, m.predict_proba(Xi_va)[:, 1]))
                    inner_train_scores.append(roc_auc_score(yi_tr, m.predict_proba(Xi_tr)[:, 1]))
                except Exception:
                    inner_val_scores.append(np.nan)
                    inner_train_scores.append(np.nan)

            avg_val   = np.nanmean(inner_val_scores)
            avg_train = np.nanmean(inner_train_scores)
            adj_score = avg_val - GAP_PENALTY * max(avg_train - avg_val, 0)

            if adj_score > best_inner_score:
                best_inner_score, best_params = adj_score, params

        # ---- Fit final model on full outer-train ----
        try:
            final_model = build_rf(best_params, RANDOM_STATE)
            final_model.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        prob_test  = final_model.predict_proba(X_test)[:, 1]
        prob_train = final_model.predict_proba(X_train)[:, 1]

        # Threshold on train only — no leakage
        thresh = find_best_threshold(y_train, prob_train, threshold_override)

        extra = {
            "Fold": fold_idx,
            "Threshold": thresh,
            "best_inner_score": float(best_inner_score),
            **{f"best_{k}": (float(v) if isinstance(v, (int, float, np.floating)) else str(v))
               for k, v in best_params.items()}
        }

        m_test  = calculate_all_metrics(y_test,  (prob_test  >= thresh).astype(int), prob_test)
        m_train = calculate_all_metrics(y_train, (prob_train >= thresh).astype(int), prob_train)
        m_test.update( {"Split": "Test",  **extra})
        m_train.update({"Split": "Train", **extra})
        records.extend([m_test, m_train])

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"depth={best_params['max_depth']}  "
            f"leaf={best_params['min_samples_leaf']}  "
            f"feat={best_params['max_features']}  "
            f"ccp={best_params.get('ccp_alpha',0.0)}  "
            f"thresh={thresh:.2f}  inner_adj={best_inner_score:.3f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Spec={m_test['Specificity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit -- consider XGBoost/LGBM/CatBoost instead"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"] and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors, threshold_override=None):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(
        X, y, label,
        inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS,
        threshold_override=threshold_override
    )


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("Random Forest | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        pos = int(grp[TARGET].sum()) if TARGET in grp.columns else 0
        neg = len(grp) - pos
        if pos > 0:
            print(f"  {name}: Positive={pos}  Negative={neg}  Ratio(neg/pos)={neg/pos:.2f}")
        else:
            print(f"  {name}: no positive cases")

    results = {}

    male_avg, male_long = analyze_group(
        males, "MALES", existing_predictors,
        threshold_override=DECISION_THRESHOLD_MALE
    )
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE", threshold_override=DECISION_THRESHOLD_MALE)
        if SAVE_RESULTS:
            male_long.to_csv("RF_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(
        females, "FEMALES", existing_predictors,
        threshold_override=DECISION_THRESHOLD_FEMALE
    )
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE", threshold_override=DECISION_THRESHOLD_FEMALE)
        if SAVE_RESULTS:
            female_long.to_csv("RF_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))
        if SAVE_RESULTS:
            summary_df.to_csv("RF_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: RF_*.csv")
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# SVM (RBF) Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
   "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False   # Set to True to save CSVs to disk

# Decision threshold — lower than 0.5 improves Sensitivity on imbalanced groups.
# Set to None for Youden-optimal auto-search per fold (on train probabilities only).
DECISION_THRESHOLD = 0.40

# Weight of the Train-Val AUC gap penalty during inner hyperparameter selection.
# Higher value -> stronger preference for low-overfit (C, gamma) combinations.
GAP_PENALTY = 0.5

# ===================== PARAM GRID (Anti-Overfitting) =====================
# Updated: removed C=0.2 and C=0.5, which were causing the Male-group
# Train-Test AUC gap to exceed 0.15 (clear overfitting in Folds 3 and 5).
# Max C is now capped at 0.1 -> larger margin, stronger regularization.
PARAM_GRID = [
    {"C": 0.001, "gamma": "scale"},
    {"C": 0.005, "gamma": "scale"},
    {"C": 0.01,  "gamma": "scale"},
    {"C": 0.01,  "gamma": 0.001},
    {"C": 0.02,  "gamma": "scale"},
    {"C": 0.02,  "gamma": 0.001},
    {"C": 0.05,  "gamma": "scale"},
    {"C": 0.05,  "gamma": 0.001},
    {"C": 0.05,  "gamma": 0.01},
    {"C": 0.1,   "gamma": 0.001},
    {"C": 0.1,   "gamma": 0.01},
]
# ==================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    """Compute comprehensive binary classification metrics."""
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]     = accuracy_score(y_true, y_pred)
    metrics["Precision"]    = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]       = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]     = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]          = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]  = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"] = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"]= jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    """Replace blank strings with NaN and coerce specified columns to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def make_pipeline(C, gamma):
    """
    Build StandardScaler + SVC(RBF) pipeline.
    Scaler is fitted inside the pipeline on train only — no data leakage.
    probability=True enables Platt-scaled calibrated probabilities.
    class_weight='balanced' compensates for class imbalance.
    """
    return Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("svc", SVC(
            kernel       = "rbf",
            probability  = True,
            class_weight = "balanced",
            C            = C,
            gamma        = gamma,
            random_state = RANDOM_STATE,
            max_iter     = 5000
        ))
    ])


def predict_proba_safe(pipe, X):
    """
    Return positive-class probability.
    Falls back to min-max scaled decision_function if predict_proba fails.
    """
    try:
        return pipe.predict_proba(X)[:, 1]
    except Exception:
        pass
    if hasattr(pipe, "decision_function"):
        scores = pipe.decision_function(X)
        smin, smax = scores.min(), scores.max()
        if smax > smin:
            return (scores - smin) / (smax - smin)
    return np.full(len(X), 0.5, dtype=float)


def find_best_threshold(y_true, y_prob):
    """
    Return DECISION_THRESHOLD if set.
    Otherwise search [0.20, 0.70] for max Youden Index on TRAIN set (no leakage).
    """
    if DECISION_THRESHOLD is not None:
        return float(DECISION_THRESHOLD)
    best_thresh, best_j = 0.5, -np.inf
    for t in np.arange(0.20, 0.71, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label):
    """Print train/test metrics side-by-side in a formatted table."""
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")
    print(f"  Decision threshold    : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden)'}")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested cross-validation with RBF-SVM + anti-overfitting regularization.

    Key design choices:
      - StandardScaler fitted inside pipeline on train only (no leakage)
      - Very small C values (0.001-0.1) enforce large margin / strong regularization
      - Small gamma values create smoother, less overfit decision boundaries
      - class_weight='balanced' handles class imbalance
      - Inner-loop model selection penalizes Train-Val AUC gap (GAP_PENALTY)
        so hyperparameters that overfit the inner-train fold are disfavored
      - DECISION_THRESHOLD < 0.5 improves Sensitivity on imbalanced groups

    Inner loop: AUC-based (C, gamma) selection, adjusted by overfit penalty.
    Outer loop: unbiased evaluation on held-out test fold.

    Returns:
        summary_dict : {"Train": {...}, "Test": {...}}
        long_df      : per-fold rows with all metrics
    """
    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  SVM RBF Anti-Overfit  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    print(f"Param combinations : {len(PARAM_GRID)}")
    print(f"Decision threshold : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden-optimal per fold)'}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        # reset_index to avoid inconsistent-sample index errors
        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # ---- Inner loop: select best (C, gamma) by AUC minus overfit penalty ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_val_scores   = []
            inner_train_scores = []

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                # reset_index on each inner fold split
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                try:
                    pipe = make_pipeline(**params)
                    pipe.fit(Xi_tr, yi_tr)
                    prob_va = predict_proba_safe(pipe, Xi_va)
                    prob_tr = predict_proba_safe(pipe, Xi_tr)
                    inner_val_scores.append(roc_auc_score(yi_va, prob_va))
                    inner_train_scores.append(roc_auc_score(yi_tr, prob_tr))
                except Exception:
                    inner_val_scores.append(np.nan)
                    inner_train_scores.append(np.nan)

            avg_val   = np.nanmean(inner_val_scores)
            avg_train = np.nanmean(inner_train_scores)
            avg_gap   = avg_train - avg_val

            # Adjusted score: reward high validation AUC, penalize large gap
            adj_score = avg_val - GAP_PENALTY * max(avg_gap, 0)

            if adj_score > best_inner_score:
                best_inner_score, best_params = adj_score, params

        # ---- Fit final model on full outer-train ----
        try:
            final_pipe = make_pipeline(**best_params)
            final_pipe.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        # Probabilities
        prob_test  = predict_proba_safe(final_pipe, X_test)
        prob_train = predict_proba_safe(final_pipe, X_train)

        # Threshold found on train only (no leakage from test)
        thresh = find_best_threshold(y_train, prob_train)

        # ---- Test metrics ----
        m_test = calculate_all_metrics(y_test, (prob_test >= thresh).astype(int), prob_test)
        m_test.update({
            "Fold": fold_idx, "Split": "Test", "Threshold": thresh,
            "best_C"    : float(best_params["C"]),
            "best_gamma": str(best_params["gamma"]),
        })
        records.append(m_test)

        # ---- Train metrics ----
        m_train = calculate_all_metrics(y_train, (prob_train >= thresh).astype(int), prob_train)
        m_train.update({
            "Fold": fold_idx, "Split": "Train", "Threshold": thresh,
            "best_C"    : float(best_params["C"]),
            "best_gamma": str(best_params["gamma"]),
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"C={best_params['C']}  gamma={best_params['gamma']}  "
            f"thresh={thresh:.2f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test":  summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    """Run full nested-CV pipeline for one sex group."""
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("SVM RBF | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
    print()
    print("Anti-overfitting techniques applied:")
    print("  1. C: 0.001-0.1           (was 0.001-0.5) -> larger margin, less overfit")
    print("  2. gamma: scale/0.001/0.01               -> smoother boundary")
    print("  3. StandardScaler inside pipeline         -> no data leakage from scaling")
    print("  4. class_weight='balanced'                -> handles class imbalance")
    print("  5. DECISION_THRESHOLD=0.40                -> improves Sensitivity")
    print("  6. Inner-CV Gap Penalty (0.5)             -> penalizes overfit params")
    print("  7. max_iter=5000                          -> ensures convergence")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        grp_c = grp.dropna(subset=[TARGET])
        pos   = int(grp_c[TARGET].sum())
        neg   = len(grp_c) - pos
        print(f"  {name}: Positive={pos}  Negative={neg}  "
              f"Ratio(neg/pos)={neg/pos:.2f}" if pos > 0 else f"  {name}: no positive cases")

    results = {}

    # ---- Males ----
    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS:
            male_long.to_csv("SVM_Male_Folds_TrainTest.csv", index=False)

    # ---- Females ----
    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS:
            female_long.to_csv("SVM_Female_Folds_TrainTest.csv", index=False)

    # ---- Combined comparison table ----
    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))
        if SAVE_RESULTS:
            summary_df.to_csv("SVM_Summary_TrainTest_bySex.csv", index=False)
            print("\nResults saved to: SVM_*.csv")
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# KNN Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# Optional: SMOTE for inner-fold oversampling (install with: pip install imbalanced-learn)
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_OK = True
except ImportError:
    SMOTE_OK = False


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
  "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

# Decision threshold strategy:
#   None  -> auto Youden-optimal per fold (RECOMMENDED — balances Sensitivity & Specificity)
#   0.35  -> fixed low  (high Recall, low Precision)
#   0.50  -> fixed high (low Recall,  high Precision/Accuracy — bad for imbalanced)
DECISION_THRESHOLD = None   # auto Youden: best balance between Recall and Precision

# Use SMOTE inside each inner fold to oversample minority class?
# IMPORTANT: SMOTE is only applied when minority/majority ratio < SMOTE_SAMPLING_RATIO.
#            If the group is already balanced (e.g. females), SMOTE is auto-skipped.
# Requires: pip install imbalanced-learn
USE_SMOTE = True

# SMOTE target ratio: minority / majority after oversampling.
# 0.6 = minority reaches 60% of majority size.
# Auto-skipped if natural ratio already exceeds this value (e.g. females).
SMOTE_SAMPLING_RATIO = 0.6

# ===================== PARAM GRID =====================
# Key changes vs original:
#   n_neighbors : 15-61  (was 5-41) -> smoother, less overfit boundary
#   weights     : "distance"        -> closer neighbors weighted more
#   p           : 1 (Manhattan)     -> often better for high-dimensional data
PARAM_GRID = [
    {"n_neighbors": 15, "weights": "distance", "p": 1},
    {"n_neighbors": 15, "weights": "distance", "p": 2},
    {"n_neighbors": 21, "weights": "distance", "p": 1},
    {"n_neighbors": 21, "weights": "distance", "p": 2},
    {"n_neighbors": 21, "weights": "uniform",  "p": 1},
    {"n_neighbors": 31, "weights": "distance", "p": 1},
    {"n_neighbors": 31, "weights": "distance", "p": 2},
    {"n_neighbors": 31, "weights": "uniform",  "p": 1},
    {"n_neighbors": 41, "weights": "distance", "p": 1},
    {"n_neighbors": 51, "weights": "distance", "p": 1},
    {"n_neighbors": 61, "weights": "distance", "p": 1},
]
# ==================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]      = accuracy_score(y_true, y_pred)
    metrics["Precision"]     = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]        = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]      = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]           = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]   = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"]  = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"] = jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def smote_needed(y_train):
    """
    Return True only if SMOTE can actually increase the minority class.
    SMOTE raises ValueError if natural minority/majority ratio >= sampling_strategy.
    For example: females have ratio ~0.61, so SMOTE with ratio=0.6 would fail.
    """
    if not (USE_SMOTE and SMOTE_OK):
        return False
    pos = int(np.sum(y_train == 1))
    neg = int(np.sum(y_train == 0))
    if neg == 0:
        return False
    natural_ratio = pos / neg
    # Only apply SMOTE if minority is genuinely underrepresented
    return natural_ratio < (SMOTE_SAMPLING_RATIO - 0.05)


def make_pipeline(n_neighbors, weights, p, apply_smote=False):
    """
    Build pipeline: StandardScaler -> (optional SMOTE) -> KNN.
    SMOTE applied INSIDE pipeline — only sees training data (no leakage).
    apply_smote=True only when smote_needed() returns True.
    """
    knn = KNeighborsClassifier(
        n_neighbors = n_neighbors,
        weights     = weights,
        p           = p,
        n_jobs      = -1
    )
    if apply_smote:
        smote = SMOTE(
            random_state      = RANDOM_STATE,
            k_neighbors       = min(5, n_neighbors - 1) if n_neighbors > 1 else 1,
            sampling_strategy = SMOTE_SAMPLING_RATIO
        )
        return ImbPipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("smote",  smote),
            ("knn",    knn)
        ])
    return Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("knn",    knn)
    ])


def predict_proba_safe(pipe, X):
    try:
        return pipe.predict_proba(X)[:, 1]
    except Exception:
        pass
    if hasattr(pipe, "decision_function"):
        scores = pipe.decision_function(X)
        smin, smax = scores.min(), scores.max()
        if smax > smin:
            return (scores - smin) / (smax - smin)
    return np.full(len(X), 0.5, dtype=float)


def find_best_threshold(y_true, y_prob):
    """
    Return DECISION_THRESHOLD if set.
    Otherwise search [0.25, 0.65] for max Youden Index on OOF probabilities (no leakage).
    Youden = Sensitivity + Specificity - 1: maximises both simultaneously.
    """
    if DECISION_THRESHOLD is not None:
        return float(DECISION_THRESHOLD)
    best_thresh, best_j = 0.5, -np.inf
    for t in np.arange(0.25, 0.66, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label, smote_applied):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY  [Train = OOF on outer-train]")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train(OOF)':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    smote_str = ("applied" if smote_applied
                 else "skipped (group already balanced)" if USE_SMOTE and SMOTE_OK
                 else "disabled (pip install imbalanced-learn)")
    print(f"\n  AUC Gap (TrainOOF - Test): {gap:.4f}  [{tag}]")
    print(f"  Decision threshold       : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden)'}")
    print(f"  SMOTE                    : {smote_str}")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested CV for KNN with fixes for class imbalance and self-neighbor artifact.

    Fix 1 — OOF train metrics:
        KNN on its own training data gives ~100% (self-neighbor).
        Train metrics are computed as OOF on outer-train via inner CV.

    Fix 2 — Adaptive SMOTE:
        Applied only when minority/majority < SMOTE_SAMPLING_RATIO.
        Auto-skipped for balanced groups (e.g. females) to avoid ValueError.

    Fix 3 — Auto Youden threshold:
        Searches [0.25, 0.65] for best Sensitivity+Specificity balance.
        Applied on OOF probabilities only (no leakage from test).

    Fix 4 — Large k + distance weighting:
        Smoother boundary, less sensitivity to local minority-class noise.
    """
    # Check if SMOTE is actually useful for this group
    apply_smote = smote_needed(y)

    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  KNN  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    nat_ratio = int(np.sum(y==1)) / max(1, int(np.sum(y==0)))
    print(f"Natural minority ratio     : {nat_ratio:.3f}  (pos/neg)")
    print(f"SMOTE                      : {'applied (ratio=' + str(SMOTE_SAMPLING_RATIO) + ')' if apply_smote else 'skipped — group already balanced'}")
    print(f"Decision threshold         : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden-optimal per fold)'}")
    print(f"Train metric               : OOF (avoids self-neighbor artifact)")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        # reset_index to avoid inconsistent-sample index errors
        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # Re-check SMOTE per fold (imbalance ratio may vary slightly across folds)
        fold_smote = smote_needed(y_train)

        # ---- Inner loop: best params by AUC ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            k = params["n_neighbors"]
            inner_scores = []
            skip = False

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                # reset_index on each inner fold split
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                if k > len(Xi_tr):
                    skip = True
                    break

                try:
                    pipe = make_pipeline(k, params["weights"], params["p"],
                                         apply_smote=fold_smote)
                    pipe.fit(Xi_tr, yi_tr)
                    inner_scores.append(
                        roc_auc_score(yi_va, predict_proba_safe(pipe, Xi_va))
                    )
                except Exception:
                    inner_scores.append(np.nan)

            if skip:
                continue

            avg_sc = np.nanmean(inner_scores)
            if avg_sc > best_inner_score:
                best_inner_score, best_params = avg_sc, params

        if best_params is None:
            best_params = {"n_neighbors": 21, "weights": "distance", "p": 1}

        # ---- Fit final model on full outer-train ----
        try:
            final_pipe = make_pipeline(
                best_params["n_neighbors"],
                best_params["weights"],
                best_params["p"],
                apply_smote=fold_smote
            )
            final_pipe.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR fitting final model: {e}")
            continue

        # ---- OOF probabilities on outer-train (avoids self-neighbor artifact) ----
        oof_prob = np.full(len(X_train), np.nan, dtype=float)

        for tr_i, val_i in inner_cv.split(X_train, y_train):
            Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
            yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
            Xi_va = X_train.iloc[val_i].reset_index(drop=True)

            try:
                pipe_oof = make_pipeline(
                    best_params["n_neighbors"],
                    best_params["weights"],
                    best_params["p"],
                    apply_smote=fold_smote
                )
                pipe_oof.fit(Xi_tr, yi_tr)
                oof_prob[val_i] = predict_proba_safe(pipe_oof, Xi_va)
            except Exception:
                pass  # NaN slots filled below

        # Fill any remaining NaN with 0.5 (safe fallback)
        oof_prob = np.where(np.isnan(oof_prob), 0.5, oof_prob)

        # Test probabilities
        prob_test = predict_proba_safe(final_pipe, X_test)

        # Threshold from OOF only — no leakage from test
        thresh = find_best_threshold(y_train, oof_prob)

        # ---- Test metrics ----
        m_test = calculate_all_metrics(
            y_test, (prob_test >= thresh).astype(int), prob_test
        )
        m_test.update({
            "Fold": fold_idx, "Split": "Test", "Threshold": thresh,
            "best_n_neighbors": int(best_params["n_neighbors"]),
            "best_weights"    : str(best_params["weights"]),
            "best_p"          : int(best_params["p"]),
            "smote_used"      : int(fold_smote),
        })
        records.append(m_test)

        # ---- Train metrics (OOF) ----
        m_train = calculate_all_metrics(
            y_train, (oof_prob >= thresh).astype(int), oof_prob
        )
        m_train.update({
            "Fold": fold_idx, "Split": "Train", "Threshold": thresh,
            "best_n_neighbors": int(best_params["n_neighbors"]),
            "best_weights"    : str(best_params["weights"]),
            "best_p"          : int(best_params["p"]),
            "smote_used"      : int(fold_smote),
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"k={best_params['n_neighbors']}  "
            f"w={best_params['weights']}  "
            f"p={best_params['p']}  "
            f"smote={'Y' if fold_smote else 'N'}  "
            f"thresh={thresh:.2f}  |  "
            f"TrainOOF AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "Overfit"
    print(f"\n  >>> Average AUC Gap (TrainOOF - Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train"       : summarize(df[df["Split"] == "Train"]),
        "Test"        : summarize(df[df["Split"] == "Test"]),
        "_smote_used" : apply_smote,
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("KNN | Anti-Overfitting + Adaptive Imbalance Fix | Nested CV | by Sex")
    print()
    if not SMOTE_OK and USE_SMOTE:
        print("  WARNING: imbalanced-learn not installed. SMOTE disabled.")
        print("  Install with: pip install imbalanced-learn")
        print()
    print("Techniques applied:")
    print("  1. OOF train metrics       -> avoids KNN self-neighbor artifact")
    print("  2. Adaptive SMOTE (ratio=0.6) -> only applied when group is imbalanced")
    print("     (auto-skipped for balanced groups like females to avoid ValueError)")
    print("  3. Large k: 15-61          -> smoother, less overfit boundary")
    print("  4. weights='distance'      -> closer neighbors weighted more")
    print("  5. Threshold=auto(Youden)  -> best Sensitivity+Specificity balance")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        grp_c = grp.dropna(subset=[TARGET])
        pos = int(grp_c[TARGET].sum())
        neg = len(grp_c) - pos
        ratio = pos / neg if neg > 0 else float("inf")
        print(f"  {name}: Positive={pos}  Negative={neg}  "
              f"Ratio(pos/neg)={ratio:.2f}  "
              f"SMOTE={'will apply' if ratio < (SMOTE_SAMPLING_RATIO - 0.05) and USE_SMOTE and SMOTE_OK else 'will skip'}")

    results = {}
    smote_flags = {}

    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"]  = male_avg["Train"]
        results["MALE_TEST"]   = male_avg["Test"]
        smote_flags["MALES"]   = male_avg.get("_smote_used", False)
        print_summary(male_avg, "MALE", smote_flags["MALES"])
        if SAVE_RESULTS:
            male_long.to_csv("KNN_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        smote_flags["FEMALES"]  = female_avg.get("_smote_used", False)
        print_summary(female_avg, "FEMALE", smote_flags["FEMALES"])
        if SAVE_RESULTS:
            female_long.to_csv("KNN_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train(OOF) vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))
        if SAVE_RESULTS:
            summary_df.to_csv("KNN_Summary_TrainTest_bySex.csv", index=False)
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# Naive Bayes Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

# Optional: SMOTE for inner-fold oversampling (pip install imbalanced-learn)
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_OK = True
except ImportError:
    SMOTE_OK = False


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

# Decision threshold:
#   None  -> auto Youden-optimal per fold (RECOMMENDED — balances Recall & Precision)
#   0.40  -> fixed moderate (use if Youden gives extreme results)
#   0.50  -> default (bad for imbalanced groups — low Recall)
DECISION_THRESHOLD = None   # auto Youden: best Sensitivity+Specificity balance

# Adaptive SMOTE: only applied when minority is genuinely underrepresented.
# Auto-skipped for balanced groups (e.g. females with pos/neg ~0.61).
USE_SMOTE            = True
SMOTE_SAMPLING_RATIO = 0.6   # target pos/neg after oversampling

# NOTE on Naive Bayes + overfitting:
#   GaussianNB has almost no regularization parameters. var_smoothing is a
#   tiny numerical stabilizer (1e-12 to 1e-6), not a real regularizer.
#   If AUC gap is large, it reflects data variability not overfitting.
#   The main tools for NB are: SMOTE + Youden threshold for Recall/Precision balance.
# ==================================================


# ===================== PARAM GRID =====================
# var_smoothing: numerical stability parameter (not regularization)
# use_scaler: StandardScaler may help NB on features with very different scales
PARAM_GRID = [
    {"var_smoothing": 1e-12, "use_scaler": True},
    {"var_smoothing": 1e-11, "use_scaler": True},
    {"var_smoothing": 1e-10, "use_scaler": True},
    {"var_smoothing": 1e-9,  "use_scaler": True},
    {"var_smoothing": 1e-8,  "use_scaler": True},
    {"var_smoothing": 1e-7,  "use_scaler": True},
    {"var_smoothing": 1e-6,  "use_scaler": True},
    {"var_smoothing": 1e-12, "use_scaler": False},
    {"var_smoothing": 1e-9,  "use_scaler": False},
    {"var_smoothing": 1e-6,  "use_scaler": False},
]
# ==================================================


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    metrics = {}
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics["Accuracy"]      = accuracy_score(y_true, y_pred)
    metrics["Precision"]     = precision_score(y_true, y_pred, zero_division=0)
    metrics["Recall"]        = recall_score(y_true, y_pred, zero_division=0)
    metrics["F1_Score"]      = f1_score(y_true, y_pred, zero_division=0)
    metrics["MCC"]           = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan
    metrics["Cohen_Kappa"]   = cohen_kappa_score(y_true, y_pred)
    metrics["Hamming_Loss"]  = hamming_loss(y_true, y_pred)
    metrics["Jaccard_Index"] = jaccard_score(y_true, y_pred, zero_division=0)

    metrics["TN"] = int(tn); metrics["FP"] = int(fp)
    metrics["FN"] = int(fn); metrics["TP"] = int(tp)

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    metrics["Sensitivity"]       = sens
    metrics["Specificity"]       = spec
    metrics["PPV"]               = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    metrics["NPV"]               = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    metrics["Youden_Index"]      = sens + spec - 1.0
    metrics["Balanced_Accuracy"] = 0.5 * (sens + spec)
    metrics["Error_Rate"]        = 1.0 - metrics["Accuracy"]

    try:    metrics["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: metrics["ROC_AUC"]    = np.nan
    try:    metrics["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: metrics["Log_Loss"]   = np.nan
    try:    metrics["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: metrics["Brier_Score"]= np.nan

    metrics["Total_Cases"]    = int(len(y_true))
    metrics["Positive_Cases"] = int(np.sum(y_true))
    metrics["Negative_Cases"] = int(len(y_true) - metrics["Positive_Cases"])
    return metrics


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def smote_needed(y):
    """
    Return True only when minority is genuinely underrepresented.
    Avoids SMOTE ValueError when natural ratio already >= SMOTE_SAMPLING_RATIO.
    Example: females pos/neg ~0.61 >= 0.6 -> auto-skipped.
    """
    if not (USE_SMOTE and SMOTE_OK):
        return False
    pos = int(np.sum(y == 1))
    neg = int(np.sum(y == 0))
    if neg == 0:
        return False
    return (pos / neg) < (SMOTE_SAMPLING_RATIO - 0.05)


def make_pipeline(var_smoothing, use_scaler, apply_smote=False):
    """
    Build pipeline: (optional StandardScaler) -> (optional SMOTE) -> GaussianNB.
    SMOTE applied INSIDE pipeline — only sees training data (no leakage).
    Note: SMOTE must come AFTER scaler (numeric features only after scaling).
    """
    nb = GaussianNB(var_smoothing=var_smoothing)

    if apply_smote and SMOTE_OK:
        smote = SMOTE(
            random_state      = RANDOM_STATE,
            sampling_strategy = SMOTE_SAMPLING_RATIO
        )
        if use_scaler:
            return ImbPipeline([
                ("scaler", StandardScaler(with_mean=True, with_std=True)),
                ("smote",  smote),
                ("nb",     nb)
            ])
        else:
            return ImbPipeline([
                ("smote", smote),
                ("nb",    nb)
            ])
    else:
        if use_scaler:
            return Pipeline([
                ("scaler", StandardScaler(with_mean=True, with_std=True)),
                ("nb",     nb)
            ])
        else:
            return Pipeline([("nb", nb)])


def predict_proba_safe(pipe, X):
    """Return positive-class probability; robust fallback if unavailable."""
    try:
        return pipe.predict_proba(X)[:, 1]
    except Exception:
        pass
    if hasattr(pipe, "decision_function"):
        scores = pipe.decision_function(X)
        smin, smax = scores.min(), scores.max()
        if smax > smin:
            return (scores - smin) / (smax - smin)
    return np.full(len(X), 0.5, dtype=float)


def find_best_threshold(y_true, y_prob):
    """
    Return DECISION_THRESHOLD if set.
    Otherwise search [0.25, 0.65] for max Youden Index on train probabilities (no leakage).
    Youden = Sensitivity + Specificity - 1: best balance between Recall and Precision.
    """
    if DECISION_THRESHOLD is not None:
        return float(DECISION_THRESHOLD)
    best_thresh, best_j = 0.5, -np.inf
    for t in np.arange(0.25, 0.66, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label, smote_applied):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit — NOTE: NB has no real regularization; gap reflects data variability"
    smote_str = ("applied" if smote_applied
                 else "skipped (group already balanced)" if USE_SMOTE and SMOTE_OK
                 else "disabled (pip install imbalanced-learn)")
    print(f"\n  AUC Gap (Train - Test): {gap:.4f}  [{tag}]")
    print(f"  Decision threshold    : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden)'}")
    print(f"  SMOTE                 : {smote_str}")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested CV for Gaussian Naive Bayes with class imbalance fixes.

    Fix 1 — Adaptive SMOTE:
        Applied only when minority/majority < SMOTE_SAMPLING_RATIO.
        Auto-skipped for balanced groups (females) to avoid SMOTE ValueError.

    Fix 2 — Auto Youden threshold:
        Searches [0.25, 0.65] for best Sensitivity+Specificity balance.
        Found on TRAIN probabilities only (no leakage from test).

    NOTE on NB overfitting:
        GaussianNB has no real regularization. var_smoothing is a numerical
        stabilizer only. AUC gap in NB reflects data variability, not overfitting.
        StandardScaler is tested as it can improve NB on heteroscedastic features.

    Returns:
        summary_dict : {"Train": {...}, "Test": {...}}
        long_df      : per-fold rows with all metrics
    """
    apply_smote = smote_needed(y)

    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  Gaussian Naive Bayes  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    nat_ratio = int(np.sum(y == 1)) / max(1, int(np.sum(y == 0)))
    print(f"Natural pos/neg ratio      : {nat_ratio:.3f}")
    print(f"SMOTE                      : {'applied (ratio=' + str(SMOTE_SAMPLING_RATIO) + ')' if apply_smote else 'skipped — group already balanced'}")
    print(f"Decision threshold         : {DECISION_THRESHOLD if DECISION_THRESHOLD else 'auto (Youden-optimal per fold)'}")
    print(f"Param combinations         : {len(PARAM_GRID)}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        # reset_index to avoid inconsistent-sample index errors
        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # Re-check SMOTE per fold
        fold_smote = smote_needed(y_train)

        # ---- Inner loop: best params by AUC ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_scores = []

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                # reset_index on each inner fold split
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                try:
                    pipe = make_pipeline(
                        params["var_smoothing"],
                        params["use_scaler"],
                        apply_smote=fold_smote
                    )
                    pipe.fit(Xi_tr, yi_tr)
                    inner_scores.append(
                        roc_auc_score(yi_va, predict_proba_safe(pipe, Xi_va))
                    )
                except Exception:
                    inner_scores.append(np.nan)

            avg_sc = np.nanmean(inner_scores)
            if avg_sc > best_inner_score:
                best_inner_score, best_params = avg_sc, params

        if best_params is None:
            best_params = {"var_smoothing": 1e-9, "use_scaler": True}

        # ---- Fit final model on full outer-train ----
        try:
            final_pipe = make_pipeline(
                best_params["var_smoothing"],
                best_params["use_scaler"],
                apply_smote=fold_smote
            )
            final_pipe.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        # Probabilities
        prob_train = predict_proba_safe(final_pipe, X_train)
        prob_test  = predict_proba_safe(final_pipe, X_test)

        # Threshold from TRAIN probabilities only (no leakage from test)
        thresh = find_best_threshold(y_train, prob_train)

        # ---- Test metrics ----
        m_test = calculate_all_metrics(
            y_test, (prob_test >= thresh).astype(int), prob_test
        )
        m_test.update({
            "Fold": fold_idx, "Split": "Test", "Threshold": thresh,
            "best_var_smoothing": float(best_params["var_smoothing"]),
            "best_use_scaler"   : int(best_params["use_scaler"]),
            "smote_used"        : int(fold_smote),
        })
        records.append(m_test)

        # ---- Train metrics ----
        m_train = calculate_all_metrics(
            y_train, (prob_train >= thresh).astype(int), prob_train
        )
        m_train.update({
            "Fold": fold_idx, "Split": "Train", "Threshold": thresh,
            "best_var_smoothing": float(best_params["var_smoothing"]),
            "best_use_scaler"   : int(best_params["use_scaler"]),
            "smote_used"        : int(fold_smote),
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"vs={best_params['var_smoothing']:.0e}  "
            f"scaler={int(best_params['use_scaler'])}  "
            f"smote={'Y' if fold_smote else 'N'}  "
            f"thresh={thresh:.2f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    avg_gap = np.nanmean(auc_gaps)
    if   avg_gap < 0.05: status = "Good"
    elif avg_gap < 0.10: status = "Acceptable"
    else:                status = "High — expected for NB (data variability, not overfitting)"
    print(f"\n  >>> Average AUC Gap (Train - Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train"       : summarize(df[df["Split"] == "Train"]),
        "Test"        : summarize(df[df["Split"] == "Test"]),
        "_smote_used" : apply_smote,
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors]
    y = df_group[TARGET].astype(int)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("Gaussian Naive Bayes | Recall/Precision Balance | Nested CV | by Sex")
    print()
    if not SMOTE_OK and USE_SMOTE:
        print("  WARNING: imbalanced-learn not installed. SMOTE disabled.")
        print("  Install with: pip install imbalanced-learn")
        print()
    print("Techniques applied:")
    print("  1. Adaptive SMOTE (ratio=0.6) -> only when group is imbalanced (males)")
    print("     auto-skipped for balanced groups (females) to avoid ValueError")
    print("  2. Threshold=auto(Youden)     -> best Sensitivity+Specificity balance")
    print("  3. StandardScaler tested      -> may improve NB on heteroscedastic features")
    print("  4. reset_index on all splits  -> prevents index mismatch errors")
    print()
    print("NOTE: GaussianNB has no real regularization (var_smoothing is numerical")
    print("      stability only). AUC gap reflects data variability, not overfitting.")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"\nWARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing values in predictors : {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing values in target     : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        grp_c = grp.dropna(subset=[TARGET])
        pos   = int(grp_c[TARGET].sum())
        neg   = len(grp_c) - pos
        ratio = pos / neg if neg > 0 else float("inf")
        will_smote = ratio < (SMOTE_SAMPLING_RATIO - 0.05) and USE_SMOTE and SMOTE_OK
        print(f"  {name}: Positive={pos}  Negative={neg}  "
              f"Ratio(pos/neg)={ratio:.2f}  "
              f"SMOTE={'will apply' if will_smote else 'will skip'}")

    results    = {}
    smote_used = {}

    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg is not None:
        results["MALE_TRAIN"]  = male_avg["Train"]
        results["MALE_TEST"]   = male_avg["Test"]
        smote_used["MALES"]    = male_avg.get("_smote_used", False)
        print_summary(male_avg, "MALE", smote_used["MALES"])
        if SAVE_RESULTS:
            male_long.to_csv("NB_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg is not None:
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        smote_used["FEMALES"]   = female_avg.get("_smote_used", False)
        print_summary(female_avg, "FEMALE", smote_used["FEMALES"])
        if SAVE_RESULTS:
            female_long.to_csv("NB_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))
        if SAVE_RESULTS:
            summary_df.to_csv("NB_Summary_TrainTest_bySex.csv", index=False)
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# ANN Model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    hamming_loss, jaccard_score, log_loss, brier_score_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# =========================================================
# DESIGN DECISIONS (why each choice was made):
#
# 1. NO sample_weight  -> not supported in this sklearn version
# 2. NO oversampling   -> duplicate rows caused overfitting (+15% gap)
# 3. NO SMOTE          -> synthetic data causes MLP to memorise artefacts
#
# The correct tools for MLP in this environment are:
#   (a) Very strong L2 regularisation (alpha 0.5-5.0)
#   (b) Very small / shallow networks
#   (c) Early stopping with generous patience
#   (d) Youden-optimal threshold per fold to recover Recall
#       without needing any data augmentation at all
# =========================================================


# ===================== CONFIG =====================
FILE_PATH    = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET       = "MetS_ATPIII"
SEX_COL      = "Sex"   # Female=0, Male=1

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

INNER_FOLDS  = 5
OUTER_FOLDS  = 5
RANDOM_STATE = 42
SAVE_RESULTS = False

# Decision threshold — the PRIMARY tool for Recall/Precision balance in this setup.
# Search range [0.25, 0.65] with Youden criterion per fold (no leakage from test).
# Youden = Sensitivity + Specificity - 1 : best joint balance.
# Set to a fixed float (e.g. 0.40) to override auto-search.
DECISION_THRESHOLD = None   # None = auto Youden per fold (RECOMMENDED)

# ===================== PARAM GRID =====================
# MLP overfitting lever: alpha (L2 weight decay).
# In MLP, alpha is the ONLY true regularisation parameter.
# Values 0.5-5.0 are extreme but necessary when:
#   - dataset is small-medium (< 2000 per group)
#   - features are correlated (dietary/metabolic data)
#   - network even with 8 neurons overfits at alpha < 0.1
#
# Networks: kept to single hidden layer (8 or 16 neurons).
# Two-layer networks removed entirely — they consistently overfitted.
#
# early_stopping=True: holds out 20% of train as internal val set.
# Training stops when val loss does not improve for 30 epochs.
# This provides a second regularisation signal independent of alpha.
PARAM_GRID = [
    # Extreme L2 — single tiny layer
    {"hidden_layer_sizes": (8,),  "alpha": 5.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (8,),  "alpha": 2.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (8,),  "alpha": 1.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (8,),  "alpha": 0.5,  "learning_rate_init": 1e-3},
    # Strong L2 — single small layer
    {"hidden_layer_sizes": (16,), "alpha": 5.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (16,), "alpha": 2.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (16,), "alpha": 1.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (16,), "alpha": 0.5,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (16,), "alpha": 0.5,  "learning_rate_init": 1e-4},
    # Moderate L2 — slightly larger single layer
    {"hidden_layer_sizes": (32,), "alpha": 5.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (32,), "alpha": 2.0,  "learning_rate_init": 1e-3},
    {"hidden_layer_sizes": (32,), "alpha": 1.0,  "learning_rate_init": 1e-3},
]
# ==================================================


# ===================== ScaledMLP =====================
class ScaledMLP:
    """
    StandardScaler -> MLPClassifier wrapper.
    Avoids sklearn Pipeline to prevent compatibility issues with older anaconda.
    No sample_weight, no oversampling — pure L2 + early stopping regularisation.
    """
    def __init__(self, hidden_layer_sizes, alpha, learning_rate_init):
        self.scaler = StandardScaler(with_mean=True, with_std=True)
        self.mlp    = MLPClassifier(
            hidden_layer_sizes  = hidden_layer_sizes,
            activation          = "relu",
            solver              = "adam",
            alpha               = alpha,
            learning_rate_init  = learning_rate_init,
            max_iter            = 300,
            early_stopping      = True,
            n_iter_no_change    = 30,    # patient: wait 30 epochs before stopping
            validation_fraction = 0.20,  # 20% of train for early-stop signal
            random_state        = RANDOM_STATE,
            shuffle             = True
        )

    def fit(self, X, y):
        self.mlp.fit(self.scaler.fit_transform(X), y)
        return self

    def predict_proba(self, X):
        return self.mlp.predict_proba(self.scaler.transform(X))


# ===================== METRICS =====================
def calculate_all_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    m = {
        "Accuracy"         : accuracy_score(y_true, y_pred),
        "Precision"        : precision_score(y_true, y_pred, zero_division=0),
        "Recall"           : recall_score(y_true, y_pred, zero_division=0),
        "F1_Score"         : f1_score(y_true, y_pred, zero_division=0),
        "MCC"              : matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan,
        "Cohen_Kappa"      : cohen_kappa_score(y_true, y_pred),
        "Hamming_Loss"     : hamming_loss(y_true, y_pred),
        "Jaccard_Index"    : jaccard_score(y_true, y_pred, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Sensitivity"      : sens,
        "Specificity"      : spec,
        "PPV"              : tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "NPV"              : tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        "Youden_Index"     : sens + spec - 1.0,
        "Balanced_Accuracy": 0.5 * (sens + spec),
        "Error_Rate"       : 1.0 - accuracy_score(y_true, y_pred),
        "Total_Cases"      : int(len(y_true)),
        "Positive_Cases"   : int(np.sum(y_true)),
        "Negative_Cases"   : int(len(y_true) - np.sum(y_true)),
    }
    try:    m["ROC_AUC"]    = roc_auc_score(y_true, y_prob)
    except: m["ROC_AUC"]    = np.nan
    try:    m["Log_Loss"]   = log_loss(y_true, y_prob, eps=1e-15)
    except: m["Log_Loss"]   = np.nan
    try:    m["Brier_Score"]= brier_score_loss(y_true, y_prob)
    except: m["Brier_Score"]= np.nan
    return m


# ===================== UTILS =====================
def clean_data(df, columns):
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = out[col].replace(r"^\s*$", np.nan, regex=True)
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def predict_proba_safe(model, X):
    try:
        return model.predict_proba(X)[:, 1]
    except Exception:
        return np.full(len(X), 0.5, dtype=float)


def find_youden_threshold(y_true, y_prob):
    """
    If DECISION_THRESHOLD is set: return it (fixed).
    Otherwise: search [0.25, 0.65] for max Youden Index.
    Computed on TRAIN probabilities only — no leakage from test.
    Youden = Sensitivity + Specificity - 1 (maximises both simultaneously).
    Search range bounded to [0.25, 0.65] to prevent extreme thresholds
    that collapse either Recall or Precision to near zero.
    """
    if DECISION_THRESHOLD is not None:
        return float(DECISION_THRESHOLD)

    best_thresh, best_j = 0.50, -np.inf
    for t in np.arange(0.25, 0.66, 0.01):
        preds = (y_prob >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j    = sens + spec - 1.0
        if j > best_j:
            best_j, best_thresh = j, float(t)
    return best_thresh


def print_summary(avg, label):
    train, test = avg["Train"], avg["Test"]
    key_metrics = [
        "Accuracy", "Precision", "Recall", "F1_Score", "MCC",
        "ROC_AUC", "Sensitivity", "Specificity", "Balanced_Accuracy",
        "Brier_Score", "Log_Loss", "Youden_Index", "PPV", "NPV",
        "Cohen_Kappa", "Jaccard_Index",
    ]
    print(f"\n{'='*80}")
    print(f"{label} SUMMARY")
    print(f"{'='*80}")
    print(f"{'Metric':<24} {'Train':>10}  {'+-Std':>9}     {'Test':>10}  {'+-Std':>9}")
    print("-" * 72)
    for k in key_metrics:
        if k in train and k in test:
            print(
                f"{k:<24} "
                f"{train[k]:>10.4f}  +-{train.get(k+'_Std', 0):>8.4f}     "
                f"{test[k]:>10.4f}  +-{test.get(k+'_Std', 0):>8.4f}"
            )
    gap = train.get("ROC_AUC", 0) - test.get("ROC_AUC", 0)
    if   gap < 0.05: tag = "Good"
    elif gap < 0.10: tag = "Acceptable"
    else:            tag = "Overfit"
    print(f"\n  AUC Gap (Train - Test) : {gap:.4f}  [{tag}]")
    print(f"  Threshold mode         : {'fixed=' + str(DECISION_THRESHOLD) if DECISION_THRESHOLD else 'auto Youden [0.25, 0.65]'}")
    print(f"\n  Tuning guide:")
    print(f"    Recall still too low  -> set DECISION_THRESHOLD = 0.35")
    print(f"    Precision still low   -> set DECISION_THRESHOLD = 0.48")
    print(f"    AUC gap still high    -> increase alpha in PARAM_GRID (e.g. add 10.0)")


# ===================== NESTED CV =====================
def nested_cv_analysis(X, y, group_name, inner_folds=5, outer_folds=5):
    """
    Nested CV for MLP/ANN.

    Regularisation strategy (no data augmentation of any kind):
      - Strong L2 via alpha=0.5-5.0 (primary overfitting lever)
      - Single hidden layer 8-32 neurons only
      - early_stopping with validation_fraction=0.20, patience=30

    Recall/Precision balance (no sample_weight, no oversampling):
      - Youden-optimal threshold found on TRAIN probs each fold
      - Bounded search [0.25, 0.65] prevents extreme thresholds

    Why no data augmentation for MLP:
      - sample_weight: not supported in this sklearn version
      - oversampling:  MLP memorises duplicates -> +15% AUC gap
      - SMOTE:         MLP memorises synthetic boundary -> overfitting
      The only safe tools here are L2 + early_stopping + threshold tuning.
    """
    print("\n" + "=" * 80)
    print(f"Analysis: {group_name}  |  MLP/ANN  |  Nested {outer_folds}x{inner_folds} CV")
    print(f"N={len(X)}   Positive={int(np.sum(y))}   Negative={int(np.sum(1-y))}")
    nat_ratio = int(np.sum(y == 1)) / max(1, int(np.sum(y == 0)))
    print(f"Natural pos/neg ratio : {nat_ratio:.3f}")
    print(f"Threshold mode        : {'fixed=' + str(DECISION_THRESHOLD) if DECISION_THRESHOLD else 'auto Youden [0.25, 0.65] per fold'}")
    print(f"Param combinations    : {len(PARAM_GRID)}")
    print("=" * 80 + "\n")

    outer_cv = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=RANDOM_STATE)

    records  = []
    auc_gaps = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

        # reset_index on every outer fold split
        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test  = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test  = y.iloc[test_idx].reset_index(drop=True)

        # ---- Inner CV: select best params by AUC ----
        best_params, best_inner_score = None, -np.inf

        for params in PARAM_GRID:
            inner_scores = []

            for tr_i, val_i in inner_cv.split(X_train, y_train):
                Xi_tr = X_train.iloc[tr_i].reset_index(drop=True)
                yi_tr = y_train.iloc[tr_i].reset_index(drop=True)
                Xi_va = X_train.iloc[val_i].reset_index(drop=True)
                yi_va = y_train.iloc[val_i].reset_index(drop=True)

                try:
                    model = ScaledMLP(
                        params["hidden_layer_sizes"],
                        params["alpha"],
                        params["learning_rate_init"]
                    )
                    model.fit(Xi_tr, yi_tr)
                    inner_scores.append(
                        roc_auc_score(yi_va, predict_proba_safe(model, Xi_va))
                    )
                except Exception:
                    inner_scores.append(np.nan)

            avg_sc = np.nanmean(inner_scores)
            if avg_sc > best_inner_score:
                best_inner_score, best_params = avg_sc, params

        if best_params is None:
            best_params = {"hidden_layer_sizes": (16,), "alpha": 2.0,
                           "learning_rate_init": 1e-3}

        # ---- Fit final model on full outer-train ----
        try:
            final_model = ScaledMLP(
                best_params["hidden_layer_sizes"],
                best_params["alpha"],
                best_params["learning_rate_init"]
            )
            final_model.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Fold {fold_idx}] ERROR: {e}")
            continue

        prob_train = predict_proba_safe(final_model, X_train)
        prob_test  = predict_proba_safe(final_model, X_test)

        # Threshold found on TRAIN probs only (no leakage)
        thresh = find_youden_threshold(y_train, prob_train)

        # ---- Test metrics ----
        m_test = calculate_all_metrics(
            y_test, (prob_test >= thresh).astype(int), prob_test
        )
        m_test.update({
            "Fold": fold_idx, "Split": "Test", "Threshold": thresh,
            "best_hidden_layers"     : str(best_params["hidden_layer_sizes"]),
            "best_alpha"             : float(best_params["alpha"]),
            "best_learning_rate_init": float(best_params["learning_rate_init"]),
        })
        records.append(m_test)

        # ---- Train metrics ----
        m_train = calculate_all_metrics(
            y_train, (prob_train >= thresh).astype(int), prob_train
        )
        m_train.update({
            "Fold": fold_idx, "Split": "Train", "Threshold": thresh,
            "best_hidden_layers"     : str(best_params["hidden_layer_sizes"]),
            "best_alpha"             : float(best_params["alpha"]),
            "best_learning_rate_init": float(best_params["learning_rate_init"]),
        })
        records.append(m_train)

        gap = m_train.get("ROC_AUC", np.nan) - m_test.get("ROC_AUC", np.nan)
        auc_gaps.append(gap)

        print(
            f"  Fold {fold_idx}/{outer_folds}  |  "
            f"hls={best_params['hidden_layer_sizes']}  "
            f"alpha={best_params['alpha']:.1f}  "
            f"lr={best_params['learning_rate_init']:.0e}  "
            f"thresh={thresh:.2f}  |  "
            f"Train AUC={m_train['ROC_AUC']:.3f}  "
            f"Test AUC={m_test['ROC_AUC']:.3f}  "
            f"Prec={m_test['Precision']:.3f}  "
            f"Sens={m_test['Sensitivity']:.3f}  "
            f"Gap={gap:.3f}"
        )

    if not records:
        print(f"\n  [ERROR] All folds failed for {group_name}.")
        return {"Train": {}, "Test": {}}, pd.DataFrame()

    avg_gap = np.nanmean(auc_gaps) if auc_gaps else np.nan
    if   np.isnan(avg_gap): status = "No valid folds"
    elif avg_gap < 0.05:    status = "Good"
    elif avg_gap < 0.10:    status = "Acceptable"
    else:                   status = "Overfit"
    print(f"\n  >>> Average AUC Gap (Train - Test): {avg_gap:.3f}  [{status}]")

    df = pd.DataFrame(records)

    def summarize(df_part):
        if df_part.empty:
            return {}
        num_cols = [
            c for c in df_part.columns
            if c not in ["Fold", "Split"]
            and pd.api.types.is_numeric_dtype(df_part[c])
        ]
        out = {}
        for c in num_cols:
            out[c]          = float(df_part[c].mean())
            out[f"{c}_Std"] = float(df_part[c].std(ddof=1))
        return out

    return {
        "Train": summarize(df[df["Split"] == "Train"]),
        "Test" : summarize(df[df["Split"] == "Test"]),
    }, df


# ===================== GROUP ANALYSIS =====================
def analyze_group(df_group, label, existing_predictors):
    if len(df_group) <= 10:
        print(f"\nInsufficient samples for {label} (n={len(df_group)}). Skipping.")
        return None, None
    needed   = [TARGET] + existing_predictors
    df_group = df_group.dropna(subset=needed).copy()
    if len(df_group) <= 10 or len(np.unique(df_group[TARGET])) < 2:
        print(f"\n{label}: Not enough clean samples or single-class target. Skipping.")
        return None, None
    X = df_group[existing_predictors].reset_index(drop=True)
    y = df_group[TARGET].astype(int).reset_index(drop=True)
    return nested_cv_analysis(X, y, label, inner_folds=INNER_FOLDS, outer_folds=OUTER_FOLDS)


# ===================== MAIN =====================
def main():
    print("=" * 80)
    print("MLP/ANN | Extreme Anti-Overfitting | Nested CV | by Sex")
    print()
    print("Strategy (no data augmentation — causes overfitting in MLP):")
    print("  1. alpha: 0.5 to 5.0           (very strong L2 — primary regulariser)")
    print("  2. Networks: (8,) or (16,) or (32,) — single layer only")
    print("  3. early_stopping=True  patience=30  val_fraction=0.20")
    print("  4. Youden threshold [0.25-0.65] — balances Recall & Precision")
    print("     without any data modification")
    print()
    print("NOT used (all cause more overfitting in MLP with this sklearn version):")
    print("  - sample_weight (not supported in this sklearn)")
    print("  - oversampling  (MLP memorises duplicates -> +15% gap)")
    print("  - SMOTE         (MLP memorises synthetic boundary)")
    print("=" * 80)

    print("\nLoading data...")
    data = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
    print(f"Data shape: {data.shape}")

    existing_predictors = [p for p in PREDICTORS if p in data.columns]
    missing_predictors  = [p for p in PREDICTORS if p not in data.columns]
    if missing_predictors:
        print(f"WARNING: Missing predictors (ignored): {missing_predictors}")
    print(f"Using {len(existing_predictors)}/{len(PREDICTORS)} predictors")

    core_missing = [c for c in [SEX_COL, TARGET] if c not in data.columns]
    if core_missing:
        raise KeyError(f"Missing core columns: {core_missing}")

    subset_cols = [SEX_COL, TARGET] + existing_predictors
    data = data[subset_cols].copy()
    data = clean_data(data, [c for c in subset_cols if c != SEX_COL])

    print(f"\nMissing in predictors: {int(data[existing_predictors].isnull().sum().sum())}")
    print(f"Missing in target    : {int(data[TARGET].isnull().sum())}")

    males   = data[data[SEX_COL] == 1].copy()
    females = data[data[SEX_COL] == 0].copy()
    print(f"\nMales  : {len(males)} samples")
    print(f"Females: {len(females)} samples")

    for grp, name in [(males, "Males"), (females, "Females")]:
        g = grp.dropna(subset=[TARGET])
        pos = int(g[TARGET].sum()); neg = len(g) - pos
        print(f"  {name}: Positive={pos}  Negative={neg}  "
              f"Ratio(pos/neg)={pos/neg:.2f}")

    results = {}

    male_avg, male_long = analyze_group(males, "MALES", existing_predictors)
    if male_avg and male_avg.get("Test"):
        results["MALE_TRAIN"] = male_avg["Train"]
        results["MALE_TEST"]  = male_avg["Test"]
        print_summary(male_avg, "MALE")
        if SAVE_RESULTS and male_long is not None and not male_long.empty:
            male_long.to_csv("ANN_Male_Folds_TrainTest.csv", index=False)

    female_avg, female_long = analyze_group(females, "FEMALES", existing_predictors)
    if female_avg and female_avg.get("Test"):
        results["FEMALE_TRAIN"] = female_avg["Train"]
        results["FEMALE_TEST"]  = female_avg["Test"]
        print_summary(female_avg, "FEMALE")
        if SAVE_RESULTS and female_long is not None and not female_long.empty:
            female_long.to_csv("ANN_Female_Folds_TrainTest.csv", index=False)

    if results:
        summary_rows = [{"Group_Split": tag, **d} for tag, d in results.items()]
        summary_df   = pd.DataFrame(summary_rows)
        display_cols = ["Group_Split", "Accuracy", "ROC_AUC", "Sensitivity",
                        "Specificity", "F1_Score", "MCC", "Balanced_Accuracy",
                        "Brier_Score", "Log_Loss"]
        display_cols = [c for c in display_cols if c in summary_df.columns]
        print(f"\n{'='*80}")
        print("COMPARISON SUMMARY -- Train vs Test per Sex")
        print(f"{'='*80}")
        print(summary_df[display_cols].fillna("").to_string(index=False))
        if SAVE_RESULTS:
            summary_df.to_csv("ANN_Summary_TrainTest_bySex.csv", index=False)
    else:
        print("\nNo valid groups to summarize.")


if __name__ == "__main__":
    main()

# LightGBM Model with(1. Calibration Curve 2. Hosmer–Lemeshow Test 3. Expected Calibration Error (ECE) 4. Reliability Diagram)

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, brier_score_loss, log_loss,
    confusion_matrix, cohen_kappa_score, jaccard_score
)
from sklearn.calibration import calibration_curve
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Configuration
# ============================================================================
FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

TARGET = "MetS_ATPIII"

PARAM_GRID = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.02, 0.03],
    'reg_lambda': [10.0, 15.0, 20.0],
    'min_child_samples': [20, 30, 40],
    'colsample_bytree': [0.5, 0.6],
}

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
RANDOM_STATE = 42

# ============================================================================
# Helper Functions
# ============================================================================

def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = precision_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': ppv,
        'Recall': sensitivity,
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'ROC_AUC': roc_auc_score(y_true, y_proba),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Balanced_Accuracy': (sensitivity + specificity) / 2,
        'Brier_Score': brier_score_loss(y_true, y_proba),
        'Log_Loss': log_loss(y_true, y_proba),
        'Youden_Index': sensitivity + specificity - 1,
        'PPV': ppv,
        'NPV': npv,
        'Cohen_Kappa': cohen_kappa_score(y_true, y_pred),
        'Jaccard_Index': jaccard_score(y_true, y_pred, zero_division=0)
    }


def hosmer_lemeshow_test(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_proba, bins[1:-1])
    observed = np.bincount(bin_indices, weights=y_true, minlength=n_bins)
    expected = np.bincount(bin_indices, weights=y_proba, minlength=n_bins)
    total = np.bincount(bin_indices, minlength=n_bins)
    mask = total > 0
    chi2 = np.sum((observed[mask] - expected[mask])**2 / (expected[mask] + 1e-10))
    p_value = 1 - stats.chi2.cdf(chi2, n_bins - 2)
    return chi2, p_value


def expected_calibration_error(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_proba, bins[1:-1])
    ece = 0
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_acc = y_true[mask].mean()
            bin_conf = y_proba[mask].mean()
            ece += mask.sum() * np.abs(bin_acc - bin_conf)
    return ece / len(y_true)


def plot_calibration_analysis(y_true, y_proba, group_name):
    hl_chi2, hl_p = hosmer_lemeshow_test(y_true, y_proba)
    ece = expected_calibration_error(y_true, y_proba)
    calibrated = "Good calibration ✓" if hl_p > 0.05 else "Poor calibration ✗"

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"LightGBM Calibration Analysis — {group_name}", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    frac_pos, mean_pred = calibration_curve(y_true, y_proba, n_bins=10)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", linewidth=1.5)
    ax.plot(mean_pred, frac_pos, "s-", color="royalblue", linewidth=2,
            markersize=8, label="LightGBM")
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.2,
                    color="royalblue", label="Calibration gap")
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_ylabel("Fraction of Positives", fontsize=11)
    ax.set_title("Calibration Curve", fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

    ax = axes[0, 1]
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        mask = (y_proba >= bin_edges[i]) & (y_proba < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_accs.append(y_true[mask].mean())
            bin_confs.append(y_proba[mask].mean())
            bin_counts.append(mask.sum())
    bin_confs = np.array(bin_confs)
    bin_accs = np.array(bin_accs)
    bin_counts = np.array(bin_counts)
    ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.6,
           color="steelblue", label="Observed fraction")
    for i in range(len(bin_confs)):
        gap_height = abs(bin_accs[i] - bin_confs[i])
        gap_bottom = min(bin_accs[i], bin_confs[i])
        ax.bar(bin_confs[i], gap_height, width=0.08, bottom=gap_bottom,
               alpha=0.4, color="salmon", edgecolor="none")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", linewidth=1.5)
    ax.scatter(bin_confs, bin_accs, color="steelblue", s=80, zorder=5,
               edgecolors="darkblue", linewidths=1.5)
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_ylabel("Fraction of Positives", fontsize=11)
    ax.set_title("Reliability Diagram", fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

    ax = axes[1, 0]
    pos_probs = y_proba[y_true == 1]
    neg_probs = y_proba[y_true == 0]
    ax.hist(neg_probs, bins=30, alpha=0.6, color="steelblue",
            label="Negative (y=0)", density=True, edgecolor="darkblue", linewidth=0.5)
    ax.hist(pos_probs, bins=30, alpha=0.6, color="salmon",
            label="Positive (y=1)", density=True, edgecolor="darkred", linewidth=0.5)
    ax.axvline(0.5, color="black", linestyle="--", linewidth=2, label="Threshold=0.5")
    ax.set_xlabel("Predicted Probability", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title("Predicted Probability Distribution", fontsize=12, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    ax = axes[1, 1]
    ax.axis('off')
    summary_text = f"""CALIBRATION METRICS SUMMARY
Group  : {group_name}
Model  : LightGBM

Hosmer-Lemeshow Test
  χ² statistic : {hl_chi2:.4f}
  p-value      : {hl_p:.4f}
  df           : 8
  Interpretation: {calibrated}

Expected Calibration Error
  ECE          : {ece:.4f}
  Quality      : {"Good" if ece < 0.1 else "Poor"}

Bins used      : 10
N samples      : {len(y_true)}
Prevalence     : {y_true.mean():.3f}"""
    ax.text(0.1, 0.5, summary_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow',
                      edgecolor='black', linewidth=1.5, alpha=0.9),
            family='monospace')

    plt.tight_layout()
    plt.show()
    print(f"\n[{group_name}] Calibration plot displayed\n")


def run_nested_cv_analysis(X_data, y_data, group_name):
    n_samples = len(y_data)
    n_positive = int(y_data.sum())
    n_negative = n_samples - n_positive
    n_params = (len(PARAM_GRID['max_depth']) *
                len(PARAM_GRID['learning_rate']) *
                len(PARAM_GRID['reg_lambda']) *
                len(PARAM_GRID['min_child_samples']) *
                len(PARAM_GRID['colsample_bytree']))

    print("=" * 80)
    print(f"Analysis: {group_name}  |  LightGBM  |  Nested {N_OUTER_FOLDS}x{N_INNER_FOLDS} CV")
    print(f"N={n_samples}   Positive={n_positive}   Negative={n_negative}")
    print(f"Param combinations: {n_params}")
    print("=" * 80)
    print()

    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    train_metrics_list, test_metrics_list = [], []
    train_aucs, test_aucs = [], []
    all_y_true, all_y_proba = [], []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_data, y_data), 1):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

        model = LGBMClassifier(
            n_estimators=1000,
            random_state=RANDOM_STATE,
            verbose=-1
        )

        grid_search = GridSearchCV(
            model, PARAM_GRID, cv=inner_cv,
            scoring='roc_auc', n_jobs=-1, verbose=0
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        y_train_proba = best_model.predict_proba(X_train)[:, 1]
        y_train_pred = best_model.predict(X_train)
        train_auc = roc_auc_score(y_train, y_train_proba)

        y_test_proba = best_model.predict_proba(X_test)[:, 1]
        y_test_pred = best_model.predict(X_test)
        test_auc = roc_auc_score(y_test, y_test_proba)
        gap = train_auc - test_auc

        print(f"  Fold {fold_idx}/{N_OUTER_FOLDS}  |  "
              f"depth={best_params['max_depth']}  "
              f"lr={best_params['learning_rate']:.2f}  "
              f"l2={best_params['reg_lambda']}  "
              f"min_leaf={best_params['min_child_samples']}  "
              f"colsample={best_params['colsample_bytree']}  |  "
              f"Train AUC={train_auc:.3f}  "
              f"Test AUC={test_auc:.3f}  "
              f"Gap={gap:.3f}")

        train_metrics_list.append(calculate_metrics(y_train, y_train_pred, y_train_proba))
        test_metrics_list.append(calculate_metrics(y_test, y_test_pred, y_test_proba))
        train_aucs.append(train_auc)
        test_aucs.append(test_auc)
        all_y_true.extend(y_test)
        all_y_proba.extend(y_test_proba)

    avg_gap = np.mean(train_aucs) - np.mean(test_aucs)
    gap_status = "Acceptable" if avg_gap < 0.1 else "High"

    print()
    print(f"  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{gap_status}]")
    print()

    all_y_true = np.array(all_y_true)
    all_y_proba = np.array(all_y_proba)

    hl_chi2, hl_p = hosmer_lemeshow_test(all_y_true, all_y_proba)
    ece = expected_calibration_error(all_y_true, all_y_proba)
    calib_status = "Good" if hl_p > 0.05 else "Poor"
    calib_symbol = "✓" if hl_p > 0.05 else "✗"
    ece_status = "Good" if ece < 0.1 else "Poor"

    print(f"  [{group_name}] H-L Test: χ²={hl_chi2:.4f}, p={hl_p:.4f} → {calib_status} calibration {calib_symbol}")
    print(f"  [{group_name}] ECE={ece:.4f} → {ece_status}")
    print()

    print("=" * 80)
    print(f"{group_name} SUMMARY")
    print("=" * 80)
    print(f"{'Metric':<30} {'Train':<15} {'Test':<15}")
    print("-" * 72)

    train_means = {k: np.mean([m[k] for m in train_metrics_list]) for k in train_metrics_list[0].keys()}
    train_stds = {k: np.std([m[k] for m in train_metrics_list]) for k in train_metrics_list[0].keys()}
    test_means = {k: np.mean([m[k] for m in test_metrics_list]) for k in test_metrics_list[0].keys()}
    test_stds = {k: np.std([m[k] for m in test_metrics_list]) for k in test_metrics_list[0].keys()}

    for metric in train_means.keys():
        print(f"{metric:<30} {train_means[metric]:.4f}  +-  {train_stds[metric]:.4f}         "
              f"{test_means[metric]:.4f}  +-  {test_stds[metric]:.4f}")

    print()
    print(f"  AUC Gap (Train - Test): {avg_gap:.4f}  [{gap_status}]")
    print()

    return test_means, test_stds, all_y_true, all_y_proba


# ============================================================================
# Main Analysis
# ============================================================================

print("=" * 80)
print("LightGBM | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
print()
print("Anti-overfitting techniques applied:")
print("  1. n_estimators=1000 with early stopping via GridSearchCV")
print("  2. L2 Regularization (reg_lambda: 10-20)")
print("  3. Lower depth (3-5)")
print("  4. min_child_samples (20-40)")
print("  5. Feature sampling (colsample_bytree: 0.5-0.6)")
print("  6. Lower learning_rate (0.01-0.03)")
print()
print("Calibration analysis added:")
print("  1. Calibration Curve")
print("  2. Hosmer-Lemeshow Test")
print("  3. Expected Calibration Error (ECE)")
print("  4. Reliability Diagram")
print("=" * 80)
print()

print("Loading data...")
try:
    data = pd.read_csv(FILE_PATH, encoding="utf-8")
except UnicodeDecodeError:
    data = pd.read_csv(FILE_PATH, encoding="latin-1")

print(f"Data shape: {data.shape}")
available_predictors = [p for p in PREDICTORS if p in data.columns]
print(f"Using {len(available_predictors)}/{len(PREDICTORS)} predictors")
print()

X = data[available_predictors].copy()
y = data[TARGET].copy()

X = X.replace(r'^\s*$', np.nan, regex=True)
y = y.replace(r'^\s*$', np.nan, regex=True)
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')

print(f"Missing values in predictors : {X.isnull().sum().sum()}")
print(f"Missing values in target     : {y.isnull().sum()}")
print()

X = X.fillna(X.median())
y = y.dropna()
X = X.loc[y.index]
data = data.loc[y.index]

# ============================================================================
# CHECK SEX COLUMN
# ============================================================================
print("=" * 80)
print("CHECKING SEX COLUMN")
print("=" * 80)

if "Sex" in data.columns:
    print(f"Sex column found!")
    print(f"Unique values in Sex column: {data['Sex'].unique()}")
    print(f"Value counts:")
    print(data['Sex'].value_counts())
    print()

    sex_values = data['Sex'].unique()

    if 1 in sex_values and 2 in sex_values:
        print("Detected encoding: 1=Male, 2=Female")
        males_idx = data[data["Sex"] == 1].index
        females_idx = data[data["Sex"] == 2].index
    elif 0 in sex_values and 1 in sex_values:
        print("Detected encoding: 0=Female, 1=Male")
        males_idx = data[data["Sex"] == 1].index
        females_idx = data[data["Sex"] == 0].index
    elif "M" in sex_values or "m" in sex_values:
        print("Detected encoding: M=Male, F=Female")
        males_idx = data[data["Sex"].str.upper() == "M"].index
        females_idx = data[data["Sex"].str.upper() == "F"].index
    else:
        print(f"Unknown encoding. Please specify manually.")
        print(f"Available values: {sex_values}")
        males_idx = data[data["Sex"] == sex_values[0]].index
        females_idx = data[data["Sex"] == sex_values[1]].index
        print(f"Using {sex_values[0]} as males and {sex_values[1]} as females")
else:
    print("ERROR: 'Sex' column not found in data!")
    print(f"Available columns: {data.columns.tolist()}")
    exit()

print(f"\nMales  : {len(males_idx)} samples")
print(f"Females: {len(females_idx)} samples")
print()

if len(males_idx) == 0:
    print("ERROR: No male samples found!")
    exit()
if len(females_idx) == 0:
    print("ERROR: No female samples found!")
    exit()

print("=" * 80)
print()

# ============================================================================
# MALE ANALYSIS
# ============================================================================
X_male = X.loc[males_idx].values
y_male = y.loc[males_idx].values
male_means, male_stds, male_y_true, male_y_proba = run_nested_cv_analysis(X_male, y_male, "MALES")
plot_calibration_analysis(male_y_true, male_y_proba, "MALES")

# ============================================================================
# FEMALE ANALYSIS
# ============================================================================
X_female = X.loc[females_idx].values
y_female = y.loc[females_idx].values
female_means, female_stds, female_y_true, female_y_proba = run_nested_cv_analysis(X_female, y_female, "FEMALES")
plot_calibration_analysis(female_y_true, female_y_proba, "FEMALES")

# ============================================================================
# FINAL COMPARISON TABLE
# ============================================================================
print("=" * 80)
print("FINAL COMPARISON: MALES vs FEMALES")
print("=" * 80)
print(f"{'Metric':<30} {'Males (Test)':<35} {'Females (Test)':<35}")
print("-" * 100)

for metric in male_means.keys():
    print(f"{metric:<30} {male_means[metric]:.4f}  +-  {male_stds[metric]:.4f}         "
          f"{female_means[metric]:.4f}  +-  {female_stds[metric]:.4f}")

print("=" * 80)
print("\nAnalysis complete! Calibration plots displayed for MALES and FEMALES.")

# Robustness & Missingness Stress Test For LightGBM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer, KNNImputer
from lightgbm import LGBMClassifier

np.random.seed(42)

# ===============================
# SETTINGS
# ===============================

FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET    = "MetS_ATPIII"
SEX_COL   = "Sex"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

N_SPLITS       = 5
MISSING_LEVELS = [0.1, 0.2, 0.3, 0.4]
NOISE_LEVELS   = [0.05, 0.1, 0.15]
SHIFT_LEVELS   = [0.5, 1.0]

GROUP_COLORS  = {"All": "#2166ac", "Male": "#d6604d", "Female": "#4dac26"}
GROUP_MARKERS = {"All": "o", "Male": "s", "Female": "^"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 13,
    "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.labelsize": 13, "axes.labelweight": "bold",
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "legend.framealpha": 0.9,
    "grid.alpha": 0.3, "grid.linestyle": "--"
})

# ===============================
# LOAD DATA
# ===============================

df_raw = pd.read_csv(FILE_PATH)
df_raw = df_raw.replace(r'^\s*$', np.nan, regex=True)
df_raw = df_raw[PREDICTORS + [TARGET, SEX_COL]]
for c in df_raw.columns:
    df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')
df_raw = df_raw.dropna()

X      = df_raw[PREDICTORS].values
y      = df_raw[TARGET].values
gender = df_raw[SEX_COL].values

print(f"Samples: {X.shape[0]}  |  Features: {X.shape[1]}")

# ===============================
# MODEL
# ===============================

def build_model():
    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        reg_lambda=1.0,
        min_child_samples=20,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

# ===============================
# PERTURBATIONS
# ===============================

def introduce_missingness(X, frac):
    X = X.copy()
    rows = np.random.randint(0, X.shape[0], int(X.size * frac))
    cols = np.random.randint(0, X.shape[1], int(X.size * frac))
    X[rows, cols] = np.nan
    return X

def add_noise(X, level):
    return X + np.random.normal(0, np.std(X, axis=0) * level, X.shape)

def distribution_shift(X, shift):
    noise = np.random.normal(0, 1, X.shape)
    return X * (1 + shift * 0.3) + shift * np.std(X, axis=0) * noise

# ===============================
# EVALUATION
# ===============================

def evaluate_auc(X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        m = build_model()
        m.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return np.mean(aucs)

# ===============================
# ROBUSTNESS ANALYSIS
# ===============================

def run_analysis(X, y, gender):
    results = []
    imputers = {
        "mean":   SimpleImputer(strategy="mean"),
        "median": SimpleImputer(strategy="median"),
        "knn":    KNNImputer(n_neighbors=5)
    }
    groups = {
        "All":    (X, y),
        "Male":   (X[gender == 1], y[gender == 1]),
        "Female": (X[gender == 0], y[gender == 0])
    }

    for g, (Xg, yg) in groups.items():
        print(f"\n{'='*30}\nGroup: {g}\n{'='*30}")

        baseline = evaluate_auc(Xg, yg)
        print(f"Baseline AUC: {baseline:.4f}")
        results.append({"Group": g, "Test": "Baseline", "Condition": "Complete", "AUC": baseline})

        for m in MISSING_LEVELS:
            for name, imp in imputers.items():
                Xm = imp.fit_transform(introduce_missingness(Xg, m))
                auc = evaluate_auc(Xm, yg)
                results.append({"Group": g, "Test": "Missing", "Condition": f"{int(m*100)}%_{name}", "AUC": auc})
                print(f"  Missing {int(m*100)}% [{name}]: {auc:.4f}")

        for n in NOISE_LEVELS:
            auc = evaluate_auc(add_noise(Xg, n), yg)
            results.append({"Group": g, "Test": "Noise", "Condition": f"noise_{n}", "AUC": auc})
            print(f"  Noise {n}: {auc:.4f}")

        for s in SHIFT_LEVELS:
            auc = evaluate_auc(distribution_shift(Xg, s), yg)
            results.append({"Group": g, "Test": "Shift", "Condition": f"shift_{s}", "AUC": auc})
            print(f"  Shift {s}: {auc:.4f}")

    return pd.DataFrame(results)

# ===============================
# PLOTS
# ===============================

def plot_results(df):

    # --- 1. Missing Data ---
    miss = df[df["Test"] == "Missing"].copy()
    miss["pct"] = miss["Condition"].str.extract(r"(\d+)%").astype(int)
    agg = miss.groupby(["Group", "pct"])["AUC"].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = agg[agg["Group"] == g].sort_values("pct")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["pct"], sub["mean"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        ax.fill_between(sub["pct"], sub["mean"]-sub["std"], sub["mean"]+sub["std"], alpha=0.15, color=c)
        last = sub.iloc[-1]
        ax.annotate(f'{last["mean"]:.3f}', (last["pct"], last["mean"]),
                    xytext=(8, 0), textcoords="offset points", fontsize=9, color=c, fontweight="bold")

    b0 = df[(df["Test"] == "Baseline") & (df["Group"] == "All")]["AUC"].values[0]
    ax.axhline(b0, color=GROUP_COLORS["All"], ls=":", lw=1.5, alpha=0.6, label=f"All Baseline ({b0:.3f})")
    ax.set(xlabel="Missing Data (%)", ylabel="AUC (mean ± std across imputers)",
           title="LightGBM — Missing Data Robustness")
    ax.set_xticks([10, 20, 30, 40])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d%%"))
    ax.legend(); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("lightgbm_missing_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 2. Noise ---
    noise = df[df["Test"] == "Noise"].copy()
    noise["lvl"] = noise["Condition"].str.extract(r"noise_(\d+\.\d+)").astype(float)

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = noise[noise["Group"] == g].sort_values("lvl")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["lvl"], sub["AUC"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(f'{row["AUC"]:.3f}', (row["lvl"], row["AUC"]),
                        xytext=(0, 10), textcoords="offset points", fontsize=9, color=c, ha="center")

    for g in ["All", "Male", "Female"]:
        b = df[(df["Test"] == "Baseline") & (df["Group"] == g)]["AUC"].values[0]
        ax.axhline(b, color=GROUP_COLORS[g], ls=":", lw=1.2, alpha=0.5)

    ax.set(xlabel="Noise Level (× feature std)", ylabel="AUC",
           title="LightGBM — Noise Robustness")
    ax.set_xticks([0.05, 0.10, 0.15])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("lightgbm_noise_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 3. Distribution Shift ---
    shift = df[df["Test"] == "Shift"].copy()
    shift["lvl"] = shift["Condition"].str.extract(r"shift_(\d+\.\d+)").astype(float)
    base_rows = df[df["Test"] == "Baseline"][["Group", "AUC"]].copy()
    base_rows["lvl"] = 0.0
    shift = pd.concat([base_rows, shift[["Group", "AUC", "lvl"]]], ignore_index=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = shift[shift["Group"] == g].sort_values("lvl")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["lvl"], sub["AUC"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(f'{row["AUC"]:.3f}', (row["lvl"], row["AUC"]),
                        xytext=(0, 10), textcoords="offset points", fontsize=9, color=c, ha="center")

    ax.set(xlabel="Distribution Shift Level (scale + noise)", ylabel="AUC",
           title="LightGBM — Distribution Shift Robustness")
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    ax.legend(loc="lower left"); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("lightgbm_shift_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 4. Summary Heatmap ---
    agg_h = df.groupby(["Group", "Test", "Condition"])["AUC"].mean().reset_index()
    pivot = agg_h.pivot_table(values="AUC", index="Condition", columns="Group")[["All", "Male", "Female"]]

    test_map = df.drop_duplicates("Condition").set_index("Condition")["Test"]
    order = {"Baseline": 0, "Missing": 1, "Noise": 2, "Shift": 3}
    sorted_index = sorted(pivot.index, key=lambda x: (order.get(test_map.get(x, ""), 9), x))
    pivot = pivot.loc[sorted_index]

    pivot_T = pivot.T

    fig, ax = plt.subplots(figsize=(max(12, len(pivot_T.columns) * 0.75), 4))
    sns.heatmap(pivot_T, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.70, vmax=0.85,
                linewidths=0.5, linecolor="#dddddd", ax=ax, annot_kws={"size": 9})
    ax.set(title="LightGBM Robustness — Full Summary Heatmap", xlabel="Condition", ylabel="Group")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("lightgbm_heatmap_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

# ===============================
# RUN
# ===============================

print("\n" + "="*60)
print("LightGBM Robustness Analysis")
print("="*60)

results = run_analysis(X, y, gender)
results.to_csv("lightgbm_robustness_results.csv", index=False)
print("\nSaved: lightgbm_robustness_results.csv")

plot_results(results)

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(results.groupby(["Group", "Test"])["AUC"].describe())

print("\n" + "="*60)
print("ROBUSTNESS COMPARISON BY GROUP")
print("="*60)
for group in ["All", "Male", "Female"]:
    gd = results[results["Group"] == group]
    baseline = gd[gd["Test"] == "Baseline"]["AUC"].values[0]
    print(f"\n{group}:  Baseline AUC = {baseline:.4f}")
    for test in ["Missing", "Noise", "Shift"]:
        td = gd[gd["Test"] == test]
        if len(td):
            print(f"  {test:10s} — Mean: {td['AUC'].mean():.4f}  "
                  f"Min: {td['AUC'].min():.4f}  "
                  f"Max Drop: {baseline - td['AUC'].min():.4f}")

print("\n" + "="*60)
print("Analysis complete!")
print("="*60)

# SHAP Stability Analysis For LightGBM model

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import spearmanr

# ── Configuration ─────────────────────────────────────────────────────────────
FILE_PATH  = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET     = "MetS_ATPIII"
SEX_COL    = "Sex"

PREDICTORS = [
   "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

# Model builder parameters (without early_stopping / eval_metric — these two are applied in fit())
PARAM_GRID = {
    "n_estimators":     500,
    "max_depth":        6,
    "learning_rate":     0.03,
    "reg_lambda":        5,
    "min_child_samples": 30,
    "max_bin":           128,
    "objective":        "binary",
    "verbosity":        -1,
    "n_jobs":           -1,
}

EARLY_STOPPING_ROUNDS = 50
EVAL_METRIC           = "auc"

MODEL_LABEL      = "LightGBM"
RANDOM_STATE     = 42
N_SPLITS         = 3
N_SHAP_SEEDS     = 6
TOP_N            = 15
NOISE_LEVEL      = 0.02
MISS_RATE        = 0.05
BOOTSTRAP_ITER   = 200
SHAP_SAMPLE_SIZE = 800

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 11, "figure.dpi": 150
})


# ── Core SHAP Function ────────────────────────────────────────────────────────
def get_shap_importance(X_tr, y_tr, X_te, y_te, params, seed):
    p = {**params, "random_state": seed}
    model = LGBMClassifier(**p)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        eval_metric=EVAL_METRIC,
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
    )

    X_sample = (X_te.sample(SHAP_SAMPLE_SIZE, random_state=seed)
                if len(X_te) > SHAP_SAMPLE_SIZE else X_te)

    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    if isinstance(sv, list):
        sv = sv[1]

    return np.abs(sv).mean(axis=0), model


def get_shap_values_full(model, X_tr, X_te):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_te)
    if isinstance(sv, list):
        sv = sv[1]
    ev = explainer.expected_value
    if isinstance(ev, (list, np.ndarray)):
        ev = ev[1]
    return sv, ev


# ── Main Analysis ─────────────────────────────────────────────────────────────
def shap_stability_analysis(df_input, group_label, predictor_cols):

    print(f"\n{'='*70}")
    print(f"  SHAP STABILITY ANALYSIS — {group_label} ({MODEL_LABEL})")
    print(f"{'='*70}")

    X = df_input[predictor_cols].reset_index(drop=True)
    y = df_input[TARGET].reset_index(drop=True)
    feat_names = np.array(predictor_cols)

    # ── Part 1: Fold Stability ────────────────────────────────────────────────
    print("\n── Part 1: SHAP Stability Across Folds")
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_imps = []

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        imp, _ = get_shap_importance(
            X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te],
            PARAM_GRID, RANDOM_STATE + fold
        )
        fold_imps.append(imp)

    fold_imps  = np.array(fold_imps)
    imp_mean   = fold_imps.mean(axis=0)
    imp_std    = fold_imps.std(axis=0)
    rank_order = np.argsort(-imp_mean)

    fold_ranks = np.argsort(np.argsort(-fold_imps, axis=1), axis=1).astype(float) + 1
    rank_mean  = fold_ranks.mean(axis=0)
    rank_std   = fold_ranks.std(axis=0)
    cv         = 100 * rank_std / (rank_mean + 1e-9)
    avg_cv     = cv.mean()

    print(f"  Average Rank-CV% = {avg_cv:.2f}%")
    print(f"\n  {'Feature':<26} {'Mean':>10} {'Std':>10} {'RankCV%':>10}")
    print(f"  {'-'*58}")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {imp_mean[i]:>10.5f} "
              f"{imp_std[i]:>10.5f} {cv[i]:>10.2f}")

    # ── Part 1B: Bootstrap Stability ─────────────────────────────────────────
    print("\n── Part 1B: Bootstrap SHAP Stability")
    boot_imps = []
    rng = np.random.default_rng(RANDOM_STATE)

    for i in range(BOOTSTRAP_ITER):
        idx = rng.choice(len(X), len(X), replace=True)
        X_b, y_b = X.iloc[idx], y.iloc[idx]
        try:
            X_bt, X_bv, y_bt, y_bv = train_test_split(
                X_b, y_b, test_size=0.2, stratify=y_b, random_state=i
            )
            imp, _ = get_shap_importance(
                X_bt, y_bt, X_bv, y_bv, PARAM_GRID, RANDOM_STATE + i
            )
            boot_imps.append(imp)
        except Exception:
            continue

    boot_imps  = np.array(boot_imps)
    boot_mean  = boot_imps.mean(axis=0)
    boot_low   = np.percentile(boot_imps, 2.5,  axis=0)
    boot_high  = np.percentile(boot_imps, 97.5, axis=0)
    boot_ranks = np.argsort(np.argsort(-boot_imps, axis=1), axis=1) + 1
    rank_low   = np.percentile(boot_ranks, 2.5,  axis=0)
    rank_high  = np.percentile(boot_ranks, 97.5, axis=0)

    print(f"\n  {'Feature':<26} {'Mean':>10} {'CI 2.5%':>10} {'CI 97.5%':>10}")
    print(f"  {'-'*58}")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {boot_mean[i]:>10.5f} "
              f"{boot_low[i]:>10.5f} {boot_high[i]:>10.5f}")

    print("\n  Rank Confidence Intervals (95% CI)")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {rank_low[i]:6.1f} – {rank_high[i]:6.1f}")

    # ── Part 2: Seed Stability ────────────────────────────────────────────────
    print("\n── Part 2: Seed Stability")
    tr_idx, te_idx = next(
        StratifiedKFold(N_SPLITS, shuffle=True,
                        random_state=RANDOM_STATE).split(X, y)
    )
    X_tr_s, X_te_s = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr_s, y_te_s = y.iloc[tr_idx], y.iloc[te_idx]

    seeds     = [RANDOM_STATE + k * 17 for k in range(N_SHAP_SEEDS)]
    seed_imps = np.array([
        get_shap_importance(X_tr_s, y_tr_s, X_te_s, y_te_s, PARAM_GRID, s)[0]
        for s in seeds
    ])

    rhos = [
        spearmanr(seed_imps[i], seed_imps[j]).statistic
        for i in range(len(seed_imps))
        for j in range(i + 1, len(seed_imps))
    ]
    mean_rho = np.mean(rhos)
    print(f"  Mean Spearman ρ across seeds = {mean_rho:.3f}")

    # ── Part 3: Perturbation Stability ───────────────────────────────────────
    print("\n── Part 3: Perturbation Stability")
    imp_clean, final_model = get_shap_importance(
        X_tr_s, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    feat_std = X_tr_s.std(axis=0)
    X_noise  = X_tr_s.copy()
    X_noise  = X_noise + np.random.default_rng(RANDOM_STATE).normal(
        0, (feat_std * NOISE_LEVEL).values, X_tr_s.shape
    )
    imp_noise, _ = get_shap_importance(
        X_noise, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    X_miss = X_tr_s.copy().astype(float)
    mask   = np.random.default_rng(RANDOM_STATE).random(X_miss.shape) < MISS_RATE
    X_miss[mask] = np.nan
    X_miss = X_miss.fillna(X_miss.median())
    imp_miss, _ = get_shap_importance(
        X_miss, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    rho_noise = spearmanr(imp_clean, imp_noise).statistic
    rho_miss  = spearmanr(imp_clean, imp_miss).statistic
    print(f"  Noise perturbation   ρ = {rho_noise:.3f}")
    print(f"  Missing data         ρ = {rho_miss:.3f}")

    # ── Full SHAP values ──────────────────────────────────────────────────────
    sv_full, expected_val = get_shap_values_full(final_model, X_tr_s, X_te_s)

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Stability Dashboard (2×2)
    # ════════════════════════════════════════════════════════════════════════
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"SHAP Stability Analysis — {group_label} ({MODEL_LABEL})",
        fontsize=15, fontweight="bold", y=1.01
    )

    # [0,0] Fold importance bar
    ax = axs[0, 0]
    top_idx = rank_order[:TOP_N][::-1]
    ax.barh(feat_names[top_idx], imp_mean[top_idx],
            xerr=imp_std[top_idx], color="#4472C4",
            ecolor="#888", capsize=3, height=0.6)
    ax.set_title(f"Fold SHAP Importance — Top {TOP_N} (mean ± std)")
    ax.set_xlabel("Mean |SHAP Value|")
    ax.grid(axis="x", alpha=0.3)

    # [0,1] Seed rank lines
    ax = axs[0, 1]
    seed_ranks = np.argsort(np.argsort(-seed_imps, axis=1), axis=1) + 1
    for i in range(N_SHAP_SEEDS):
        ax.plot(feat_names, seed_ranks[i], alpha=0.55, marker="o", ms=3)
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names, rotation=90, fontsize=7)
    ax.set_ylabel("Feature Rank")
    ax.set_title(f"Seed Rank Stability  (mean ρ = {mean_rho:.3f})")
    ax.grid(alpha=0.3)

    # [1,0] Perturbation bar
    ax = axs[1, 0]
    p_labels = ["Clean", "Noise", "Missing"]
    p_values = [1.0, rho_noise, rho_miss]
    p_colors = [
        "#70AD47",
        "#70AD47" if rho_noise >= 0.85 else "#ED7D31",
        "#70AD47" if rho_miss  >= 0.85 else "#FF6B6B",
    ]
    bars = ax.bar(p_labels, p_values, color=p_colors, width=0.5)
    ax.axhline(0.85, color="gray", ls="--", lw=1.2, label="Threshold 0.85")
    for b, v in zip(bars, p_values):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01,
                f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Spearman ρ")
    ax.set_title("Perturbation Stability")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # [1,1] Heatmap
    ax = axs[1, 1]
    sns.heatmap(
        fold_imps[:, rank_order[:TOP_N]],
        xticklabels=feat_names[rank_order[:TOP_N]],
        yticklabels=[f"Fold {i+1}" for i in range(N_SPLITS)],
        cmap="YlOrRd", ax=ax, linewidths=0.3
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_title(f"Fold × Feature SHAP Heatmap — Top {TOP_N}")

    plt.tight_layout()
    plt.savefig(f"fig1_stability_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 2 — SHAP Beeswarm
    # ════════════════════════════════════════════════════════════════════════
    shap.summary_plot(sv_full, X_te_s, max_display=TOP_N, show=False)
    plt.title(f"SHAP Beeswarm — {group_label} ({MODEL_LABEL})",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig2_beeswarm_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 3 — Bootstrap Rank CI
    # ════════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(figsize=(10, 6))
    top_boot = rank_order[:TOP_N]
    y_pos    = np.arange(TOP_N)
    med_rank = np.median(boot_ranks[:, top_boot], axis=0)

    ax.barh(y_pos,
            rank_high[top_boot] - rank_low[top_boot],
            left=rank_low[top_boot],
            color="#A8C4E0", alpha=0.8, height=0.6, label="95% CI")
    ax.plot(med_rank, y_pos, "o", color="#1A3A6B", zorder=5, ms=6, label="Median Rank")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_names[top_boot])
    ax.set_xlabel("Feature Rank")
    ax.set_title(
        f"Bootstrap Rank Confidence Intervals — Top {TOP_N}\n"
        f"{group_label} ({MODEL_LABEL})"
    )
    ax.legend()
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"fig3_boot_rank_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 4 — SHAP Dependence Plots (Top 3)
    # ════════════════════════════════════════════════════════════════════════
    top3 = rank_order[:3]
    fig, axs4 = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f"SHAP Dependence Plots (Top 3) — {group_label} ({MODEL_LABEL})",
        fontsize=13, fontweight="bold"
    )
    for ax, feat_idx in zip(axs4, top3):
        shap.dependence_plot(
            feat_names[feat_idx], sv_full, X_te_s,
            ax=ax, show=False, dot_size=15
        )
        ax.set_title(feat_names[feat_idx])
    plt.tight_layout()
    plt.savefig(f"fig4_dependence_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 5 — Waterfall Plot (highest-risk sample)
    # ════════════════════════════════════════════════════════════════════════
    risk_idx = int(np.argmax(sv_full.sum(axis=1)))
    shap.waterfall_plot(
        shap.Explanation(
            values=sv_full[risk_idx],
            base_values=expected_val,
            data=X_te_s.iloc[risk_idx].values,
            feature_names=list(feat_names)
        ),
        max_display=TOP_N,
        show=False
    )
    plt.title(
        f"SHAP Waterfall (Highest-Risk Sample) — {group_label} ({MODEL_LABEL})",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"fig5_waterfall_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ── Final Stability Report ────────────────────────────────────────────────
    def status(val, good, warn):
        if val <= good: return "STABLE ✓"
        if val <= warn: return "MOD. STABLE ⚠"
        return "UNSTABLE ✗"

    fold_status  = status(avg_cv, 15, 30)
    seed_status  = ("HIGHLY STABLE ✓" if mean_rho >= 0.90
                    else "MOD. STABLE ⚠" if mean_rho >= 0.75 else "UNSTABLE ✗")
    noise_status = "STABLE ✓" if rho_noise >= 0.85 else "UNSTABLE ⚠"
    miss_status  = "STABLE ✓" if rho_miss  >= 0.85 else "UNSTABLE ⚠"

    print(f"\n{'='*60}")
    print(f"  FINAL STABILITY REPORT — {group_label} ({MODEL_LABEL})")
    print(f"{'='*60}")
    print(f"  Fold Stability     : {fold_status:<20}  (Rank-CV = {avg_cv:.1f}%)")
    print(f"  Seed Stability     : {seed_status:<20}  (ρ = {mean_rho:.3f})")
    print(f"  Noise Robustness   : {noise_status:<20}  (ρ = {rho_noise:.3f})")
    print(f"  Missing Robustness : {miss_status:<20}  (ρ = {rho_miss:.3f})")
    print(f"{'='*60}\n")


# ── Entry Point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv(FILE_PATH)
    df[PREDICTORS] = df[PREDICTORS].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=PREDICTORS + [TARGET, SEX_COL])

    shap_stability_analysis(df[df[SEX_COL] == 0], "FEMALE", PREDICTORS)
    shap_stability_analysis(df[df[SEX_COL] == 1], "MALE",   PREDICTORS)

# EBM Model with(1. Calibration Curve 2. Hosmer–Lemeshow Test 3. Expected Calibration Error (ECE) 4. Reliability Diagram)

In [ ]:
import pandas as pd
import numpy as np
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, brier_score_loss, log_loss,
    confusion_matrix, cohen_kappa_score, jaccard_score
)
from sklearn.calibration import calibration_curve
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Configuration
# ============================================================================
FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

TARGET = "MetS_ATPIII"

PARAM_GRID = {
    'max_leaves': [3, 4, 5],
    'learning_rate': [0.01, 0.02, 0.03],
    'l2_regularization': [10.0, 15.0, 20.0],
    'min_samples_leaf': [20, 30, 40],
    'max_bins': [128, 256],
}

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
RANDOM_STATE = 42

# ============================================================================
# Helper Functions
# ============================================================================

def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = precision_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': ppv,
        'Recall': sensitivity,
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'ROC_AUC': roc_auc_score(y_true, y_proba),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Balanced_Accuracy': (sensitivity + specificity) / 2,
        'Brier_Score': brier_score_loss(y_true, y_proba),
        'Log_Loss': log_loss(y_true, y_proba),
        'Youden_Index': sensitivity + specificity - 1,
        'PPV': ppv,
        'NPV': npv,
        'Cohen_Kappa': cohen_kappa_score(y_true, y_pred),
        'Jaccard_Index': jaccard_score(y_true, y_pred, zero_division=0)
    }


def hosmer_lemeshow_test(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_proba, bins[1:-1])
    observed = np.bincount(bin_indices, weights=y_true, minlength=n_bins)
    expected = np.bincount(bin_indices, weights=y_proba, minlength=n_bins)
    total = np.bincount(bin_indices, minlength=n_bins)
    mask = total > 0
    chi2 = np.sum((observed[mask] - expected[mask])**2 / (expected[mask] + 1e-10))
    p_value = 1 - stats.chi2.cdf(chi2, n_bins - 2)
    return chi2, p_value


def expected_calibration_error(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_proba, bins[1:-1])
    ece = 0
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_acc = y_true[mask].mean()
            bin_conf = y_proba[mask].mean()
            ece += mask.sum() * np.abs(bin_acc - bin_conf)
    return ece / len(y_true)


def plot_calibration_analysis(y_true, y_proba, group_name):
    hl_chi2, hl_p = hosmer_lemeshow_test(y_true, y_proba)
    ece = expected_calibration_error(y_true, y_proba)
    calibrated = "Good calibration ✓" if hl_p > 0.05 else "Poor calibration ✗"

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"EBM Calibration Analysis — {group_name}", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    frac_pos, mean_pred = calibration_curve(y_true, y_proba, n_bins=10)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", linewidth=1.5)
    ax.plot(mean_pred, frac_pos, "s-", color="royalblue", linewidth=2,
            markersize=8, label="EBM")
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.2,
                    color="royalblue", label="Calibration gap")
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_ylabel("Fraction of Positives", fontsize=11)
    ax.set_title("Calibration Curve", fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

    ax = axes[0, 1]
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        mask = (y_proba >= bin_edges[i]) & (y_proba < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_accs.append(y_true[mask].mean())
            bin_confs.append(y_proba[mask].mean())
            bin_counts.append(mask.sum())
    bin_confs = np.array(bin_confs)
    bin_accs = np.array(bin_accs)
    bin_counts = np.array(bin_counts)
    ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.6,
           color="steelblue", label="Observed fraction")
    for i in range(len(bin_confs)):
        gap_height = abs(bin_accs[i] - bin_confs[i])
        gap_bottom = min(bin_accs[i], bin_confs[i])
        ax.bar(bin_confs[i], gap_height, width=0.08, bottom=gap_bottom,
               alpha=0.4, color="salmon", edgecolor="none")
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", linewidth=1.5)
    ax.scatter(bin_confs, bin_accs, color="steelblue", s=80, zorder=5,
               edgecolors="darkblue", linewidths=1.5)
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax.set_ylabel("Fraction of Positives", fontsize=11)
    ax.set_title("Reliability Diagram", fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

    ax = axes[1, 0]
    pos_probs = y_proba[y_true == 1]
    neg_probs = y_proba[y_true == 0]
    ax.hist(neg_probs, bins=30, alpha=0.6, color="steelblue",
            label="Negative (y=0)", density=True, edgecolor="darkblue", linewidth=0.5)
    ax.hist(pos_probs, bins=30, alpha=0.6, color="salmon",
            label="Positive (y=1)", density=True, edgecolor="darkred", linewidth=0.5)
    ax.axvline(0.5, color="black", linestyle="--", linewidth=2, label="Threshold=0.5")
    ax.set_xlabel("Predicted Probability", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title("Predicted Probability Distribution", fontsize=12, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    ax = axes[1, 1]
    ax.axis('off')
    summary_text = f"""CALIBRATION METRICS SUMMARY
Group  : {group_name}
Model  : EBM

Hosmer-Lemeshow Test
  χ² statistic : {hl_chi2:.4f}
  p-value      : {hl_p:.4f}
  df           : 8
  Interpretation: {calibrated}

Expected Calibration Error
  ECE          : {ece:.4f}
  Quality      : {"Good" if ece < 0.1 else "Poor"}

Bins used      : 10
N samples      : {len(y_true)}
Prevalence     : {y_true.mean():.3f}"""
    ax.text(0.1, 0.5, summary_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow',
                      edgecolor='black', linewidth=1.5, alpha=0.9),
            family='monospace')

    plt.tight_layout()
    plt.show()
    print(f"\n[{group_name}] Calibration plot displayed\n")


def run_nested_cv_analysis(X_data, y_data, group_name):
    n_samples = len(y_data)
    n_positive = int(y_data.sum())
    n_negative = n_samples - n_positive
    n_params = (len(PARAM_GRID['max_leaves']) *
                len(PARAM_GRID['learning_rate']) *
                len(PARAM_GRID['l2_regularization']) *
                len(PARAM_GRID['min_samples_leaf']) *
                len(PARAM_GRID['max_bins']))

    print("=" * 80)
    print(f"Analysis: {group_name}  |  EBM  |  Nested {N_OUTER_FOLDS}x{N_INNER_FOLDS} CV")
    print(f"N={n_samples}   Positive={n_positive}   Negative={n_negative}")
    print(f"Param combinations: {n_params}")
    print("=" * 80)
    print()

    outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    train_metrics_list, test_metrics_list = [], []
    train_aucs, test_aucs = [], []
    all_y_true, all_y_proba = [], []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_data, y_data), 1):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

        model = ExplainableBoostingClassifier(
            random_state=RANDOM_STATE
        )

        grid_search = GridSearchCV(
            model, PARAM_GRID, cv=inner_cv,
            scoring='roc_auc', n_jobs=-1, verbose=0
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        y_train_proba = best_model.predict_proba(X_train)[:, 1]
        y_train_pred = best_model.predict(X_train)
        train_auc = roc_auc_score(y_train, y_train_proba)

        y_test_proba = best_model.predict_proba(X_test)[:, 1]
        y_test_pred = best_model.predict(X_test)
        test_auc = roc_auc_score(y_test, y_test_proba)
        gap = train_auc - test_auc

        print(f"  Fold {fold_idx}/{N_OUTER_FOLDS}  |  "
              f"leaves={best_params['max_leaves']}  "
              f"lr={best_params['learning_rate']:.2f}  "
              f"l2={best_params['l2_regularization']}  "
              f"min_leaf={best_params['min_samples_leaf']}  "
              f"bins={best_params['max_bins']}  |  "
              f"Train AUC={train_auc:.3f}  "
              f"Test AUC={test_auc:.3f}  "
              f"Gap={gap:.3f}")

        train_metrics_list.append(calculate_metrics(y_train, y_train_pred, y_train_proba))
        test_metrics_list.append(calculate_metrics(y_test, y_test_pred, y_test_proba))
        train_aucs.append(train_auc)
        test_aucs.append(test_auc)
        all_y_true.extend(y_test)
        all_y_proba.extend(y_test_proba)

    avg_gap = np.mean(train_aucs) - np.mean(test_aucs)
    gap_status = "Acceptable" if avg_gap < 0.1 else "High"

    print()
    print(f"  >>> Average AUC Gap (Train-Test): {avg_gap:.3f}  [{gap_status}]")
    print()

    all_y_true = np.array(all_y_true)
    all_y_proba = np.array(all_y_proba)

    hl_chi2, hl_p = hosmer_lemeshow_test(all_y_true, all_y_proba)
    ece = expected_calibration_error(all_y_true, all_y_proba)
    calib_status = "Good" if hl_p > 0.05 else "Poor"
    calib_symbol = "✓" if hl_p > 0.05 else "✗"
    ece_status = "Good" if ece < 0.1 else "Poor"

    print(f"  [{group_name}] H-L Test: χ²={hl_chi2:.4f}, p={hl_p:.4f} → {calib_status} calibration {calib_symbol}")
    print(f"  [{group_name}] ECE={ece:.4f} → {ece_status}")
    print()

    print("=" * 80)
    print(f"{group_name} SUMMARY")
    print("=" * 80)
    print(f"{'Metric':<30} {'Train':<15} {'Test':<15}")
    print("-" * 72)

    train_means = {k: np.mean([m[k] for m in train_metrics_list]) for k in train_metrics_list[0].keys()}
    train_stds = {k: np.std([m[k] for m in train_metrics_list]) for k in train_metrics_list[0].keys()}
    test_means = {k: np.mean([m[k] for m in test_metrics_list]) for k in test_metrics_list[0].keys()}
    test_stds = {k: np.std([m[k] for m in test_metrics_list]) for k in test_metrics_list[0].keys()}

    for metric in train_means.keys():
        print(f"{metric:<30} {train_means[metric]:.4f}  +-  {train_stds[metric]:.4f}         "
              f"{test_means[metric]:.4f}  +-  {test_stds[metric]:.4f}")

    print()
    print(f"  AUC Gap (Train - Test): {avg_gap:.4f}  [{gap_status}]")
    print()

    return test_means, test_stds, all_y_true, all_y_proba


# ============================================================================
# Main Analysis
# ============================================================================

print("=" * 80)
print("EBM | Anti-Overfitting | Nested CV | Train & Test Evaluation by Sex")
print()
print("Anti-overfitting techniques applied:")
print("  1. L2 Regularization (l2_regularization: 10-20)")
print("  2. Lower max_leaves (3-5)")
print("  3. min_samples_leaf (20-40)")
print("  4. Binning control (max_bins: 128-256)")
print("  5. Lower learning_rate (0.01-0.03)")
print()
print("Calibration analysis added:")
print("  1. Calibration Curve")
print("  2. Hosmer-Lemeshow Test")
print("  3. Expected Calibration Error (ECE)")
print("  4. Reliability Diagram")
print("=" * 80)
print()

print("Loading data...")
try:
    data = pd.read_csv(FILE_PATH, encoding="utf-8")
except UnicodeDecodeError:
    data = pd.read_csv(FILE_PATH, encoding="latin-1")

print(f"Data shape: {data.shape}")
available_predictors = [p for p in PREDICTORS if p in data.columns]
print(f"Using {len(available_predictors)}/{len(PREDICTORS)} predictors")
print()

X = data[available_predictors].copy()
y = data[TARGET].copy()

X = X.replace(r'^\s*$', np.nan, regex=True)
y = y.replace(r'^\s*$', np.nan, regex=True)
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')

print(f"Missing values in predictors : {X.isnull().sum().sum()}")
print(f"Missing values in target     : {y.isnull().sum()}")
print()

X = X.fillna(X.median())
y = y.dropna()
X = X.loc[y.index]
data = data.loc[y.index]

# ============================================================================
# CHECK SEX COLUMN
# ============================================================================
print("=" * 80)
print("CHECKING SEX COLUMN")
print("=" * 80)

if "Sex" in data.columns:
    print(f"Sex column found!")
    print(f"Unique values in Sex column: {data['Sex'].unique()}")
    print(f"Value counts:")
    print(data['Sex'].value_counts())
    print()

    sex_values = data['Sex'].unique()

    if 1 in sex_values and 2 in sex_values:
        print("Detected encoding: 1=Male, 2=Female")
        males_idx = data[data["Sex"] == 1].index
        females_idx = data[data["Sex"] == 2].index
    elif 0 in sex_values and 1 in sex_values:
        print("Detected encoding: 0=Female, 1=Male")
        males_idx = data[data["Sex"] == 1].index
        females_idx = data[data["Sex"] == 0].index
    elif "M" in sex_values or "m" in sex_values:
        print("Detected encoding: M=Male, F=Female")
        males_idx = data[data["Sex"].str.upper() == "M"].index
        females_idx = data[data["Sex"].str.upper() == "F"].index
    else:
        print(f"Unknown encoding. Please specify manually.")
        print(f"Available values: {sex_values}")
        males_idx = data[data["Sex"] == sex_values[0]].index
        females_idx = data[data["Sex"] == sex_values[1]].index
        print(f"Using {sex_values[0]} as males and {sex_values[1]} as females")
else:
    print("ERROR: 'Sex' column not found in data!")
    print(f"Available columns: {data.columns.tolist()}")
    exit()

print(f"\nMales  : {len(males_idx)} samples")
print(f"Females: {len(females_idx)} samples")
print()

if len(males_idx) == 0:
    print("ERROR: No male samples found!")
    exit()
if len(females_idx) == 0:
    print("ERROR: No female samples found!")
    exit()

print("=" * 80)
print()

# ============================================================================
# MALE ANALYSIS
# ============================================================================
X_male = X.loc[males_idx].values
y_male = y.loc[males_idx].values
male_means, male_stds, male_y_true, male_y_proba = run_nested_cv_analysis(X_male, y_male, "MALES")
plot_calibration_analysis(male_y_true, male_y_proba, "MALES")

# ============================================================================
# FEMALE ANALYSIS
# ============================================================================
X_female = X.loc[females_idx].values
y_female = y.loc[females_idx].values
female_means, female_stds, female_y_true, female_y_proba = run_nested_cv_analysis(X_female, y_female, "FEMALES")
plot_calibration_analysis(female_y_true, female_y_proba, "FEMALES")

# ============================================================================
# FINAL COMPARISON TABLE
# ============================================================================
print("=" * 80)
print("FINAL COMPARISON: MALES vs FEMALES")
print("=" * 80)
print(f"{'Metric':<30} {'Males (Test)':<35} {'Females (Test)':<35}")
print("-" * 100)

for metric in male_means.keys():
    print(f"{metric:<30} {male_means[metric]:.4f}  +-  {male_stds[metric]:.4f}         "
          f"{female_means[metric]:.4f}  +-  {female_stds[metric]:.4f}")

print("=" * 80)
print("\nAnalysis complete! Calibration plots displayed for MALES and FEMALES.")

# Robustness & Missingness Stress Test For EBM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer, KNNImputer
from interpret.glassbox import ExplainableBoostingClassifier

np.random.seed(42)

# ===============================
# SETTINGS
# ===============================

FILE_PATH = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET    = "MetS_ATPIII"
SEX_COL   = "Sex"

PREDICTORS = [
    "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

N_SPLITS       = 5
MISSING_LEVELS = [0.1, 0.2, 0.3, 0.4]
NOISE_LEVELS   = [0.05, 0.1, 0.15]
SHIFT_LEVELS   = [0.5, 1.0]

GROUP_COLORS  = {"All": "#2166ac", "Male": "#d6604d", "Female": "#4dac26"}
GROUP_MARKERS = {"All": "o", "Male": "s", "Female": "^"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 13,
    "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.labelsize": 13, "axes.labelweight": "bold",
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "legend.framealpha": 0.9,
    "grid.alpha": 0.3, "grid.linestyle": "--"
})

# ===============================
# LOAD DATA
# ===============================

df_raw = pd.read_csv(FILE_PATH)
df_raw = df_raw.replace(r'^\s*$', np.nan, regex=True)
df_raw = df_raw[PREDICTORS + [TARGET, SEX_COL]]
for c in df_raw.columns:
    df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')
df_raw = df_raw.dropna()

X      = df_raw[PREDICTORS].values
y      = df_raw[TARGET].values
gender = df_raw[SEX_COL].values

print(f"Samples: {X.shape[0]}  |  Features: {X.shape[1]}")

# ===============================
# MODEL
# ===============================

def build_model():
    return ExplainableBoostingClassifier(
        max_leaves=4,
        learning_rate=0.03,
        l2_regularization=15.0,
        min_samples_leaf=20,
        max_bins=256,
        random_state=42
    )

# ===============================
# PERTURBATIONS
# ===============================

def introduce_missingness(X, frac):
    X = X.copy()
    rows = np.random.randint(0, X.shape[0], int(X.size * frac))
    cols = np.random.randint(0, X.shape[1], int(X.size * frac))
    X[rows, cols] = np.nan
    return X

def add_noise(X, level):
    return X + np.random.normal(0, np.std(X, axis=0) * level, X.shape)

def distribution_shift(X, shift):
    noise = np.random.normal(0, 1, X.shape)
    return X * (1 + shift * 0.3) + shift * np.std(X, axis=0) * noise

# ===============================
# EVALUATION
# ===============================

def evaluate_auc(X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        m = build_model()
        m.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return np.mean(aucs)

# ===============================
# ROBUSTNESS ANALYSIS
# ===============================

def run_analysis(X, y, gender):
    results = []
    imputers = {
        "mean":   SimpleImputer(strategy="mean"),
        "median": SimpleImputer(strategy="median"),
        "knn":    KNNImputer(n_neighbors=5)
    }
    groups = {
        "All":    (X, y),
        "Male":   (X[gender == 1], y[gender == 1]),
        "Female": (X[gender == 0], y[gender == 0])
    }

    for g, (Xg, yg) in groups.items():
        print(f"\n{'='*30}\nGroup: {g}\n{'='*30}")

        baseline = evaluate_auc(Xg, yg)
        print(f"Baseline AUC: {baseline:.4f}")
        results.append({"Group": g, "Test": "Baseline", "Condition": "Complete", "AUC": baseline})

        for m in MISSING_LEVELS:
            for name, imp in imputers.items():
                Xm = imp.fit_transform(introduce_missingness(Xg, m))
                auc = evaluate_auc(Xm, yg)
                results.append({"Group": g, "Test": "Missing", "Condition": f"{int(m*100)}%_{name}", "AUC": auc})
                print(f"  Missing {int(m*100)}% [{name}]: {auc:.4f}")

        for n in NOISE_LEVELS:
            auc = evaluate_auc(add_noise(Xg, n), yg)
            results.append({"Group": g, "Test": "Noise", "Condition": f"noise_{n}", "AUC": auc})
            print(f"  Noise {n}: {auc:.4f}")

        for s in SHIFT_LEVELS:
            auc = evaluate_auc(distribution_shift(Xg, s), yg)
            results.append({"Group": g, "Test": "Shift", "Condition": f"shift_{s}", "AUC": auc})
            print(f"  Shift {s}: {auc:.4f}")

    return pd.DataFrame(results)

# ===============================
# PLOTS
# ===============================

def plot_results(df):

    # --- 1. Missing Data ---
    miss = df[df["Test"] == "Missing"].copy()
    miss["pct"] = miss["Condition"].str.extract(r"(\d+)%").astype(int)
    agg = miss.groupby(["Group", "pct"])["AUC"].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = agg[agg["Group"] == g].sort_values("pct")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["pct"], sub["mean"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        ax.fill_between(sub["pct"], sub["mean"]-sub["std"], sub["mean"]+sub["std"], alpha=0.15, color=c)
        last = sub.iloc[-1]
        ax.annotate(f'{last["mean"]:.3f}', (last["pct"], last["mean"]),
                    xytext=(8, 0), textcoords="offset points", fontsize=9, color=c, fontweight="bold")

    b0 = df[(df["Test"] == "Baseline") & (df["Group"] == "All")]["AUC"].values[0]
    ax.axhline(b0, color=GROUP_COLORS["All"], ls=":", lw=1.5, alpha=0.6, label=f"All Baseline ({b0:.3f})")
    ax.set(xlabel="Missing Data (%)", ylabel="AUC (mean ± std across imputers)",
           title="EBM — Missing Data Robustness")
    ax.set_xticks([10, 20, 30, 40])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d%%"))
    ax.legend(); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("ebm_missing_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 2. Noise ---
    noise = df[df["Test"] == "Noise"].copy()
    noise["lvl"] = noise["Condition"].str.extract(r"noise_(\d+\.\d+)").astype(float)

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = noise[noise["Group"] == g].sort_values("lvl")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["lvl"], sub["AUC"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(f'{row["AUC"]:.3f}', (row["lvl"], row["AUC"]),
                        xytext=(0, 10), textcoords="offset points", fontsize=9, color=c, ha="center")

    for g in ["All", "Male", "Female"]:
        b = df[(df["Test"] == "Baseline") & (df["Group"] == g)]["AUC"].values[0]
        ax.axhline(b, color=GROUP_COLORS[g], ls=":", lw=1.2, alpha=0.5)

    ax.set(xlabel="Noise Level (× feature std)", ylabel="AUC",
           title="EBM — Noise Robustness")
    ax.set_xticks([0.05, 0.10, 0.15])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("ebm_noise_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 3. Distribution Shift ---
    shift = df[df["Test"] == "Shift"].copy()
    shift["lvl"] = shift["Condition"].str.extract(r"shift_(\d+\.\d+)").astype(float)
    base_rows = df[df["Test"] == "Baseline"][["Group", "AUC"]].copy()
    base_rows["lvl"] = 0.0
    shift = pd.concat([base_rows, shift[["Group", "AUC", "lvl"]]], ignore_index=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    for g in ["All", "Male", "Female"]:
        sub = shift[shift["Group"] == g].sort_values("lvl")
        c, mk = GROUP_COLORS[g], GROUP_MARKERS[g]
        ax.plot(sub["lvl"], sub["AUC"], marker=mk, color=c, lw=2.2, ms=9, label=g, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(f'{row["AUC"]:.3f}', (row["lvl"], row["AUC"]),
                        xytext=(0, 10), textcoords="offset points", fontsize=9, color=c, ha="center")

    ax.set(xlabel="Distribution Shift Level (scale + noise)", ylabel="AUC",
           title="EBM — Distribution Shift Robustness")
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    ax.legend(loc="lower left"); ax.grid(True); sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig("ebm_shift_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- 4. Summary Heatmap ---
    agg_h = df.groupby(["Group", "Test", "Condition"])["AUC"].mean().reset_index()
    pivot = agg_h.pivot_table(values="AUC", index="Condition", columns="Group")[["All", "Male", "Female"]]

    test_map = df.drop_duplicates("Condition").set_index("Condition")["Test"]
    order = {"Baseline": 0, "Missing": 1, "Noise": 2, "Shift": 3}
    sorted_index = sorted(pivot.index, key=lambda x: (order.get(test_map.get(x, ""), 9), x))
    pivot = pivot.loc[sorted_index]

    pivot_T = pivot.T

    fig, ax = plt.subplots(figsize=(max(12, len(pivot_T.columns) * 0.75), 4))
    sns.heatmap(pivot_T, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.70, vmax=0.85,
                linewidths=0.5, linecolor="#dddddd", ax=ax, annot_kws={"size": 9})
    ax.set(title="EBM Robustness — Full Summary Heatmap", xlabel="Condition", ylabel="Group")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("ebm_heatmap_robustness.png", dpi=200, bbox_inches="tight")
    plt.show()

# ===============================
# RUN
# ===============================

print("\n" + "="*60)
print("EBM Robustness Analysis")
print("="*60)

results = run_analysis(X, y, gender)
results.to_csv("ebm_robustness_results.csv", index=False)
print("\nSaved: ebm_robustness_results.csv")

plot_results(results)

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(results.groupby(["Group", "Test"])["AUC"].describe())

print("\n" + "="*60)
print("ROBUSTNESS COMPARISON BY GROUP")
print("="*60)
for group in ["All", "Male", "Female"]:
    gd = results[results["Group"] == group]
    baseline = gd[gd["Test"] == "Baseline"]["AUC"].values[0]
    print(f"\n{group}:  Baseline AUC = {baseline:.4f}")
    for test in ["Missing", "Noise", "Shift"]:
        td = gd[gd["Test"] == test]
        if len(td):
            print(f"  {test:10s} — Mean: {td['AUC'].mean():.4f}  "
                  f"Min: {td['AUC'].min():.4f}  "
                  f"Max Drop: {baseline - td['AUC'].min():.4f}")

print("\n" + "="*60)
print("Analysis complete!")
print("="*60)

# SHAP For ExplainableBoostingClassifier- EBM

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import spearmanr

# ── Configuration ─────────────────────────────────────────────────────────────
FILE_PATH  = r"C:\Users\DELL NOAVAR\Desktop\Amol.Data.csv"
TARGET     = "MetS_ATPIII"
SEX_COL    = "Sex"

PREDICTORS = [
   "age","CVDs","cholestrol","CRP","Alt","Ast","GGT","Alkp","LDL",
    "BUN","Cr","vit.D.ug","Family_history_chronic","Blood_r","heart_r","diabet_r","past_current_smoke",
    "alcohol_drinker","MET","Residual_areas","carbohydrate_percent","protein_percent","Fat_percent",
    "fiber_per_1000kcal","NTraditi","NPrudent","NMixed_p"
]

PARAM_GRID = {
    "max_leaves":        4,
    "learning_rate":     0.03,
    "l2_regularization": 15.0,
    "min_samples_leaf":  20,
    "max_bins":          256,
}

MODEL_LABEL      = "EBM"
RANDOM_STATE     = 42
N_SPLITS         = 3
N_SHAP_SEEDS     = 6
TOP_N            = 15
NOISE_LEVEL      = 0.02
MISS_RATE        = 0.05
BOOTSTRAP_ITER   = 200
SHAP_SAMPLE_SIZE = 800

plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 11, "figure.dpi": 150
})


# ── Core SHAP Function ────────────────────────────────────────────────────────
def get_shap_importance(X_tr, y_tr, X_te, y_te, params, seed):
    p = {**params, "random_state": seed}
    model = ExplainableBoostingClassifier(**p)
    model.fit(X_tr, y_tr)

    X_sample = (X_te.sample(SHAP_SAMPLE_SIZE, random_state=seed)
                if len(X_te) > SHAP_SAMPLE_SIZE else X_te)

    explainer = shap.Explainer(model.predict_proba, X_sample)
    sv = explainer(X_sample).values
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    return np.abs(sv).mean(axis=0), model


def get_shap_values_full(model, X_tr, X_te):
    explainer = shap.Explainer(model.predict_proba, X_tr)
    explanation = explainer(X_te)
    sv = explanation.values
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    ev = explanation.base_values
    if ev.ndim == 2:
        ev = ev[:, 1]
    return sv, ev[0]


# ── Main Analysis ─────────────────────────────────────────────────────────────
def shap_stability_analysis(df_input, group_label, predictor_cols):

    print(f"\n{'='*70}")
    print(f"  SHAP STABILITY ANALYSIS — {group_label} ({MODEL_LABEL})")
    print(f"{'='*70}")

    X = df_input[predictor_cols].reset_index(drop=True)
    y = df_input[TARGET].reset_index(drop=True)
    feat_names = np.array(predictor_cols)

    # ── Part 1: Fold Stability ────────────────────────────────────────────────
    print("\n── Part 1: SHAP Stability Across Folds")
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_imps = []

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        imp, _ = get_shap_importance(
            X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te],
            PARAM_GRID, RANDOM_STATE + fold
        )
        fold_imps.append(imp)

    fold_imps  = np.array(fold_imps)
    imp_mean   = fold_imps.mean(axis=0)
    imp_std    = fold_imps.std(axis=0)
    rank_order = np.argsort(-imp_mean)

    fold_ranks = np.argsort(np.argsort(-fold_imps, axis=1), axis=1).astype(float) + 1
    rank_mean  = fold_ranks.mean(axis=0)
    rank_std   = fold_ranks.std(axis=0)
    cv         = 100 * rank_std / (rank_mean + 1e-9)
    avg_cv     = cv.mean()

    print(f"  Average Rank-CV% = {avg_cv:.2f}%")
    print(f"\n  {'Feature':<26} {'Mean':>10} {'Std':>10} {'RankCV%':>10}")
    print(f"  {'-'*58}")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {imp_mean[i]:>10.5f} "
              f"{imp_std[i]:>10.5f} {cv[i]:>10.2f}")

    # ── Part 1B: Bootstrap Stability ─────────────────────────────────────────
    print("\n── Part 1B: Bootstrap SHAP Stability")
    boot_imps = []
    rng = np.random.default_rng(RANDOM_STATE)

    for i in range(BOOTSTRAP_ITER):
        idx = rng.choice(len(X), len(X), replace=True)
        X_b, y_b = X.iloc[idx], y.iloc[idx]
        try:
            X_bt, X_bv, y_bt, y_bv = train_test_split(
                X_b, y_b, test_size=0.2, stratify=y_b, random_state=i
            )
            imp, _ = get_shap_importance(
                X_bt, y_bt, X_bv, y_bv, PARAM_GRID, RANDOM_STATE + i
            )
            boot_imps.append(imp)
        except Exception:
            continue

    boot_imps  = np.array(boot_imps)
    boot_mean  = boot_imps.mean(axis=0)
    boot_low   = np.percentile(boot_imps, 2.5,  axis=0)
    boot_high  = np.percentile(boot_imps, 97.5, axis=0)
    boot_ranks = np.argsort(np.argsort(-boot_imps, axis=1), axis=1) + 1
    rank_low   = np.percentile(boot_ranks, 2.5,  axis=0)
    rank_high  = np.percentile(boot_ranks, 97.5, axis=0)

    print(f"\n  {'Feature':<26} {'Mean':>10} {'CI 2.5%':>10} {'CI 97.5%':>10}")
    print(f"  {'-'*58}")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {boot_mean[i]:>10.5f} "
              f"{boot_low[i]:>10.5f} {boot_high[i]:>10.5f}")

    print("\n  Rank Confidence Intervals (95% CI)")
    for i in rank_order[:TOP_N]:
        print(f"  {feat_names[i]:<26} {rank_low[i]:6.1f} – {rank_high[i]:6.1f}")

    # ── Part 2: Seed Stability ────────────────────────────────────────────────
    print("\n── Part 2: Seed Stability")
    tr_idx, te_idx = next(
        StratifiedKFold(N_SPLITS, shuffle=True,
                        random_state=RANDOM_STATE).split(X, y)
    )
    X_tr_s, X_te_s = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr_s, y_te_s = y.iloc[tr_idx], y.iloc[te_idx]

    seeds     = [RANDOM_STATE + k * 17 for k in range(N_SHAP_SEEDS)]
    seed_imps = np.array([
        get_shap_importance(X_tr_s, y_tr_s, X_te_s, y_te_s, PARAM_GRID, s)[0]
        for s in seeds
    ])

    rhos = [
        spearmanr(seed_imps[i], seed_imps[j]).statistic
        for i in range(len(seed_imps))
        for j in range(i + 1, len(seed_imps))
    ]
    mean_rho = np.mean(rhos)
    print(f"  Mean Spearman ρ across seeds = {mean_rho:.3f}")

    # ── Part 3: Perturbation Stability ───────────────────────────────────────
    print("\n── Part 3: Perturbation Stability")
    imp_clean, final_model = get_shap_importance(
        X_tr_s, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    feat_std = X_tr_s.std(axis=0)
    X_noise  = X_tr_s.copy()
    X_noise  = X_noise + np.random.default_rng(RANDOM_STATE).normal(
        0, (feat_std * NOISE_LEVEL).values, X_tr_s.shape
    )
    imp_noise, _ = get_shap_importance(
        X_noise, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    X_miss = X_tr_s.copy().astype(float)
    mask   = np.random.default_rng(RANDOM_STATE).random(X_miss.shape) < MISS_RATE
    X_miss[mask] = np.nan
    X_miss = X_miss.fillna(X_miss.median())
    imp_miss, _ = get_shap_importance(
        X_miss, y_tr_s, X_te_s, y_te_s, PARAM_GRID, RANDOM_STATE
    )

    rho_noise = spearmanr(imp_clean, imp_noise).statistic
    rho_miss  = spearmanr(imp_clean, imp_miss).statistic
    print(f"  Noise perturbation   ρ = {rho_noise:.3f}")
    print(f"  Missing data         ρ = {rho_miss:.3f}")

    # ── Full SHAP values ──────────────────────────────────────────────────────
    sv_full, expected_val = get_shap_values_full(final_model, X_tr_s, X_te_s)

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Stability Dashboard (2×2)
    # ════════════════════════════════════════════════════════════════════════
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"SHAP Stability Analysis — {group_label} ({MODEL_LABEL})",
        fontsize=15, fontweight="bold", y=1.01
    )

    ax = axs[0, 0]
    top_idx = rank_order[:TOP_N][::-1]
    ax.barh(feat_names[top_idx], imp_mean[top_idx],
            xerr=imp_std[top_idx], color="#4472C4",
            ecolor="#888", capsize=3, height=0.6)
    ax.set_title(f"Fold SHAP Importance — Top {TOP_N} (mean ± std)")
    ax.set_xlabel("Mean |SHAP Value|")
    ax.grid(axis="x", alpha=0.3)

    ax = axs[0, 1]
    seed_ranks = np.argsort(np.argsort(-seed_imps, axis=1), axis=1) + 1
    for i in range(N_SHAP_SEEDS):
        ax.plot(feat_names, seed_ranks[i], alpha=0.55, marker="o", ms=3)
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names, rotation=90, fontsize=7)
    ax.set_ylabel("Feature Rank")
    ax.set_title(f"Seed Rank Stability  (mean ρ = {mean_rho:.3f})")
    ax.grid(alpha=0.3)

    ax = axs[1, 0]
    p_labels = ["Clean", "Noise", "Missing"]
    p_values = [1.0, rho_noise, rho_miss]
    p_colors = [
        "#70AD47",
        "#70AD47" if rho_noise >= 0.85 else "#ED7D31",
        "#70AD47" if rho_miss  >= 0.85 else "#FF6B6B",
    ]
    bars = ax.bar(p_labels, p_values, color=p_colors, width=0.5)
    ax.axhline(0.85, color="gray", ls="--", lw=1.2, label="Threshold 0.85")
    for b, v in zip(bars, p_values):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01,
                f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Spearman ρ")
    ax.set_title("Perturbation Stability")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    ax = axs[1, 1]
    sns.heatmap(
        fold_imps[:, rank_order[:TOP_N]],
        xticklabels=feat_names[rank_order[:TOP_N]],
        yticklabels=[f"Fold {i+1}" for i in range(N_SPLITS)],
        cmap="YlOrRd", ax=ax, linewidths=0.3
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_title(f"Fold × Feature SHAP Heatmap — Top {TOP_N}")

    plt.tight_layout()
    plt.savefig(f"fig1_stability_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 2 — SHAP Beeswarm
    # ════════════════════════════════════════════════════════════════════════
    shap.summary_plot(sv_full, X_te_s, max_display=TOP_N, show=False)
    plt.title(f"SHAP Beeswarm — {group_label} ({MODEL_LABEL})",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig2_beeswarm_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 3 — Bootstrap Rank CI
    # ════════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(figsize=(10, 6))
    top_boot = rank_order[:TOP_N]
    y_pos    = np.arange(TOP_N)
    med_rank = np.median(boot_ranks[:, top_boot], axis=0)

    ax.barh(y_pos,
            rank_high[top_boot] - rank_low[top_boot],
            left=rank_low[top_boot],
            color="#A8C4E0", alpha=0.8, height=0.6, label="95% CI")
    ax.plot(med_rank, y_pos, "o", color="#1A3A6B", zorder=5, ms=6, label="Median Rank")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_names[top_boot])
    ax.set_xlabel("Feature Rank")
    ax.set_title(
        f"Bootstrap Rank Confidence Intervals — Top {TOP_N}\n"
        f"{group_label} ({MODEL_LABEL})"
    )
    ax.legend()
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"fig3_boot_rank_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 4 — SHAP Dependence Plots (Top 3)
    # ════════════════════════════════════════════════════════════════════════
    top3 = rank_order[:3]
    fig, axs4 = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        f"SHAP Dependence Plots (Top 3) — {group_label} ({MODEL_LABEL})",
        fontsize=13, fontweight="bold"
    )
    for ax, feat_idx in zip(axs4, top3):
        shap.dependence_plot(
            feat_names[feat_idx], sv_full, X_te_s,
            ax=ax, show=False, dot_size=15
        )
        ax.set_title(feat_names[feat_idx])
    plt.tight_layout()
    plt.savefig(f"fig4_dependence_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 5 — Waterfall Plot (highest-risk sample)
    # ════════════════════════════════════════════════════════════════════════
    risk_idx = int(np.argmax(sv_full.sum(axis=1)))
    shap.waterfall_plot(
        shap.Explanation(
            values=sv_full[risk_idx],
            base_values=expected_val,
            data=X_te_s.iloc[risk_idx].values,
            feature_names=list(feat_names)
        ),
        max_display=TOP_N,
        show=False
    )
    plt.title(
        f"SHAP Waterfall (Highest-Risk Sample) — {group_label} ({MODEL_LABEL})",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(f"fig5_waterfall_{group_label}_{MODEL_LABEL}.png",
                dpi=200, bbox_inches="tight")
    plt.show()

    # ── Final Stability Report ────────────────────────────────────────────────
    def status(val, good, warn):
        if val <= good: return "STABLE ✓"
        if val <= warn: return "MOD. STABLE ⚠"
        return "UNSTABLE ✗"

    fold_status  = status(avg_cv, 15, 30)
    seed_status  = ("HIGHLY STABLE ✓" if mean_rho >= 0.90
                    else "MOD. STABLE ⚠" if mean_rho >= 0.75 else "UNSTABLE ✗")
    noise_status = "STABLE ✓" if rho_noise >= 0.85 else "UNSTABLE ⚠"
    miss_status  = "STABLE ✓" if rho_miss  >= 0.85 else "UNSTABLE ⚠"

    print(f"\n{'='*60}")
    print(f"  FINAL STABILITY REPORT — {group_label} ({MODEL_LABEL})")
    print(f"{'='*60}")
    print(f"  Fold Stability     : {fold_status:<20}  (Rank-CV = {avg_cv:.1f}%)")
    print(f"  Seed Stability     : {seed_status:<20}  (ρ = {mean_rho:.3f})")
    print(f"  Noise Robustness   : {noise_status:<20}  (ρ = {rho_noise:.3f})")
    print(f"  Missing Robustness : {miss_status:<20}  (ρ = {rho_miss:.3f})")
    print(f"{'='*60}\n")


# ── Entry Point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv(FILE_PATH)
    df[PREDICTORS] = df[PREDICTORS].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=PREDICTORS + [TARGET, SEX_COL])

    shap_stability_analysis(df[df[SEX_COL] == 0], "FEMALE", PREDICTORS)
    shap_stability_analysis(df[df[SEX_COL] == 1], "MALE",   PREDICTORS)